# Operator‑Precedence Interpretability — Feasibility Notebook (Phases 0–5)

**Goal of this notebook.** Decide whether the project is *plausible* **before** any expensive probing, by passing five gates **in order**:

| Gate | Phase | What it proves | Needs GPU? |
|------|-------|----------------|------------|
| **G0** | 0 | `transformer_lens` loads + hooks Llama‑3.1‑8B (or HF fallback) | yes (cheap) |
| **G1** | 1 | the controlled contribution is novel | no |
| **G2** | 2 | the stimulus is *genuinely* controlled (token‑identical, answer‑equal) | **no — CPU + tokenizer only** |
| **G3** | 3 | the model *computes* (not looks up), engages the operand, in‑band | yes (cheap) |
| **G4** | 5 | the activation‑patching instrument reproduces a *known* result | yes (cheap) |

The project is **PLAUSIBLE iff G0–G4 all pass**. **G2** and **G3** are the ones that, if they fail, mean the *science* is broken (not the tooling). Phase 4 is not a gate but underpins every patch.

---

## ⚠️ How to run this on a flaky GPU (read first)

This notebook is built to **survive GPU disconnects**. Every expensive step writes its result to a **persistent artifact directory** (`ART`) and is guarded by an `if has_artifact(...)` check.

**To resume after a disconnect:** just **re‑run the notebook top‑to‑bottom** (`Restart & Run All` works). Completed phases load their cached artifacts from disk in seconds and are skipped; only unfinished work re‑runs. The model itself is reloaded each fresh session (GPU memory can't be checkpointed) but everything derived from it is cached.

**One‑time setup before the first run:**
1. A GPU with **≥ 40 GB** (Llama‑3.1‑8B in bf16 needs ~16 GB weights + activation cache). A100‑40GB / H100 are fine; a 16 GB T4 will **not** fit.
2. Llama‑3.1‑8B is **gated** on Hugging Face — request access on the model page and set your token: `export HF_TOKEN=hf_...` (or `HUGGINGFACE_TOKEN`). The Phase 0 cell logs in with it.
3. (Colab) The checkpoint cell tries to mount Google Drive so `ART` persists across runtime resets. On a dedicated box it uses a local persistent dir under `$HOME`.

**Run order is strict** — a later gate is only meaningful if the earlier ones are green. The final dashboard cell prints the consolidated G0–G4 verdict and reconstructs it from disk even after a restart.


In [ ]:
# ============================================================================
# Phase -1 / Gate G_INFRA — CONFIG + CHECKPOINT/RESUME INFRASTRUCTURE
# ----------------------------------------------------------------------------
# This is the FIRST cell. Every later cell depends on the globals/helpers it
# defines. It is fully idempotent: safe to re-run after a GPU/kernel disconnect.
# It does NO GPU work and loads NO model (model is reloaded in Phase 0).
# ============================================================================

import os
import io
import sys
import json
import time
import pickle
import random
import datetime
import platform
import pathlib
from pathlib import Path

import numpy as np

try:
    import torch
    _HAS_TORCH = True
except Exception:  # pragma: no cover - torch must exist on the GPU box, but be safe
    torch = None
    _HAS_TORCH = False


# ----------------------------------------------------------------------------
# 1. Persistent artifact directory ART (survives kernel/GPU disconnects)
# ----------------------------------------------------------------------------
# Priority:
#   (a) Google Colab + mountable Drive  -> /content/drive/MyDrive/opprec_interp
#   (b) Local persistent dir under HOME  -> ~/opprec_interp_artifacts
# Both branches are wrapped so a failure degrades gracefully to the local path.
# (b) is the expected branch on a dedicated cloud GPU box where HOME is the
# persistent user volume; only Colab uses the Drive branch.

ART_DRIVE_DEFAULT = "/content/drive/MyDrive/opprec_interp"
ART_LOCAL_DEFAULT = str(Path.home() / "opprec_interp_artifacts")


def _in_colab() -> bool:
    """True iff we appear to be running inside a Google Colab kernel."""
    if "google.colab" in sys.modules:
        return True
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _resolve_art_dir() -> Path:
    """Pick and create a persistent artifact directory; return its Path."""
    # (a) Try Colab Drive first.
    if _in_colab():
        try:
            from google.colab import drive  # type: ignore
            # Mount is idempotent: if already mounted, this is a no-op.
            if not os.path.ismount("/content/drive"):
                drive.mount("/content/drive", force_remount=False)
            art = Path(ART_DRIVE_DEFAULT)
            art.mkdir(parents=True, exist_ok=True)
            # Confirm we can actually write (Drive sometimes mounts read-only).
            _probe = art / ".write_probe"
            _probe.write_text("ok")
            _probe.unlink(missing_ok=True)
            return art
        except Exception as e:
            print(f"[ART] Colab Drive unavailable ({type(e).__name__}: {e}); "
                  f"falling back to local persistent dir.")

    # (b) Local persistent directory under the home dir.
    art = Path(ART_LOCAL_DEFAULT)
    art.mkdir(parents=True, exist_ok=True)
    return art


ART = _resolve_art_dir()
print(f"[ART] Persistent artifact directory: {ART}")


# ----------------------------------------------------------------------------
# 6. log(msg): timestamped print   (defined early so later setup can use it)
# ----------------------------------------------------------------------------
def log(msg: str) -> None:
    """Timestamped stdout print, flushed immediately."""
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] {msg}", flush=True)


# ----------------------------------------------------------------------------
# 3. CFG dict — all knobs as PARAMETERS (no hardcoded magic deep in later cells)
# ----------------------------------------------------------------------------
def _auto_device() -> str:
    if _HAS_TORCH and torch.cuda.is_available():
        return "cuda"
    return "cpu"


_DEVICE = _auto_device()
_DTYPE = (torch.bfloat16 if _HAS_TORCH else None)  # bf16: nondeterminism recorded, not fought

CFG = {
    # --- model / runtime ---
    "model_name": "meta-llama/Llama-3.1-8B",
    "seed": 0,
    "device": _DEVICE,
    "dtype": _DTYPE,
    "dtype_name": "bfloat16",
    # bf16 matmul is nondeterministic across runs; we RECORD this, we do not fight it.
    "determinism_note": (
        "bf16 matmul accumulation order is nondeterministic on GPU; small "
        "logit jitter is expected and is recorded, not eliminated."
    ),

    # --- operand-band params (parameterized span so Phase 3 can SEARCH over it) ---
    # The task studies operand-recognition / operand-precedence in arithmetic of
    # the form  a OP (b digits) OP (c digits). Digit counts are given as inclusive
    # [min, max] ranges so Phase 3 can sweep band widths rather than hardcoding.
    "b_digits": {"min": 1, "max": 3},   # inclusive digit-count band for operand b
    "c_digits": {"min": 1, "max": 3},   # inclusive digit-count band for operand c
    "a_digits": {"min": 1, "max": 1},   # leading operand (kept narrow by default)
    "operators": ["+", "-", "*"],        # operators considered in the prompt family
    # Phase 3 search grid over band widths (each entry is a (min,max) digit band).
    "digit_band_grid": [[1, 1], [1, 2], [1, 3], [2, 3], [3, 3]],

    # --- padding sweep (left-pad lengths probed for position robustness) ---
    "pad_lengths": [0, 2, 4, 8, 16],

    # --- dataset sizing ---
    "n_per_factor": 2000,   # target examples per experimental factor / band cell
    "max_new_tokens": 8,    # generation budget when checking answers
    "batch_size": 32,       # default eval batch (later cells may override per-mem)

    # --- reproducibility / bookkeeping ---
    "tl_from_pretrained": True,  # prefer transformer_lens HookedTransformer in Phase 0
    "hf_fallback": True,         # allow HF wrapper exposing run_with_cache/run_with_hooks
}

# --- derive all paths from ART (single source of truth) ---
CFG["paths"] = {
    "art": str(ART),
    "gate_status": str(ART / "gate_status.json"),
    "dataset": str(ART / "dataset.pkl"),
    "cache": str(ART / "cache"),
    "figures": str(ART / "figures"),
    "logs": str(ART / "logs"),
}
# Make sure the derived subdirectories exist.
for _sub in ("cache", "figures", "logs"):
    Path(CFG["paths"][_sub]).mkdir(parents=True, exist_ok=True)


# ----------------------------------------------------------------------------
# 4. Seeding helper (python / numpy / torch / cuda). bf16 nondeterminism noted.
# ----------------------------------------------------------------------------
def set_all_seeds(seed: int) -> None:
    """Seed python-random, numpy, and torch (+cuda). Idempotent."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if _HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        # We deliberately do NOT force torch.use_deterministic_algorithms(True):
        # bf16 matmul nondeterminism is RECORDED (see CFG['determinism_note']),
        # not fought, because forcing determinism would change/limit kernels.
    log(f"set_all_seeds({seed}) applied "
        f"(torch={'yes' if _HAS_TORCH else 'no'}, "
        f"cuda={'yes' if (_HAS_TORCH and torch.cuda.is_available()) else 'no'}).")


set_all_seeds(CFG["seed"])


# ----------------------------------------------------------------------------
# 5. Artifact helpers — all read/write UNDER ART.
#    JSON for small/structured; pickle for tensors/arrays/objects; text for raw.
# ----------------------------------------------------------------------------
def _art_path(name: str, ext: str) -> Path:
    """Map an artifact name to ART/<name><ext>, allowing names with subdirs."""
    p = ART / (name if name.endswith(ext) else f"{name}{ext}")
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def _atomic_write_bytes(path: Path, data: bytes) -> None:
    """Write bytes atomically (tmp + os.replace) so a disconnect mid-write
    cannot leave a half-written artifact that later cells would trust.

    Hardened vs. the original:
      * try/finally unlinks a stray tmp file if os.replace() ever raises, so
        failed writes don't accumulate orphan .tmp.<pid> files in ART.
      * parent-directory fsync after os.replace makes the rename itself durable
        against a true host crash (best-effort; skipped where unsupported)."""
    tmp = path.with_suffix(path.suffix + f".tmp.{os.getpid()}")
    try:
        with open(tmp, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
        # fsync the directory entry so the rename survives a crash, not just the
        # file contents. Not all platforms allow opening a dir fd; ignore if not.
        try:
            _dfd = os.open(str(path.parent), os.O_DIRECTORY)
            try:
                os.fsync(_dfd)
            finally:
                os.close(_dfd)
        except (OSError, AttributeError):
            pass
    finally:
        # If we crashed before/at os.replace, tmp may still exist — clean it up.
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


# Map the public 'kind' names to their on-disk extensions so existence checks
# can be matched to the loader that will actually be used.
_EXT_BY_KIND = {"json": ".json", "pickle": ".pkl", "text": ".txt"}


def has_artifact(name: str, kind: str = None) -> bool:
    """True iff an artifact with this base name exists on disk.

    IMPORTANT: pass `kind` ('json' | 'pickle' | 'text') to check for the SAME
    type you will load with, e.g.

        if has_artifact('cache', 'pickle'): cache = load_pickle('cache')

    Without `kind` this returns True if ANY of {.json,.pkl,.txt} exists, which
    is convenient but UNSAFE in the standard resilience idiom: an artifact saved
    via save_pickle would make `has_artifact('x')` True while `load_json('x')`
    raises FileNotFoundError. Prefer the kind-aware form to pair the existence
    check with its loader."""
    if kind is not None:
        if kind not in _EXT_BY_KIND:
            raise ValueError(f"has_artifact kind must be one of {sorted(_EXT_BY_KIND)}, got {kind!r}")
        return _art_path(name, _EXT_BY_KIND[kind]).exists()
    for ext in (".json", ".pkl", ".txt"):
        if _art_path(name, ext).exists():
            return True
    return False


def save_json(name: str, obj) -> str:
    path = _art_path(name, ".json")
    data = json.dumps(obj, indent=2, default=str).encode("utf-8")
    _atomic_write_bytes(path, data)
    return str(path)


def load_json(name: str):
    path = _art_path(name, ".json")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_pickle(name: str, obj) -> str:
    path = _art_path(name, ".pkl")
    _atomic_write_bytes(path, pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL))
    return str(path)


def load_pickle(name: str):
    path = _art_path(name, ".pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def save_text(name: str, s: str) -> str:
    path = _art_path(name, ".txt")
    _atomic_write_bytes(path, str(s).encode("utf-8"))
    return str(path)


def load_text(name: str) -> str:
    path = _art_path(name, ".txt")
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


# ----------------------------------------------------------------------------
# 7. GATE-STATUS ledger persisted to ART/gate_status.json
#    Lets the final dashboard reconstruct G0..G4 across sessions.
# ----------------------------------------------------------------------------
_GATE_FILE = Path(CFG["paths"]["gate_status"])


def get_gates() -> dict:
    """Return the full gate ledger {gate: {passed, detail, ts}}; {} if none."""
    if _GATE_FILE.exists():
        try:
            with open(_GATE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception as e:
            log(f"[gates] WARNING: could not read gate ledger ({e}); treating as empty.")
            return {}
    return {}


def set_gate(gate: str, passed: bool, detail: str = "") -> dict:
    """Record a gate result (read-modify-write the on-disk ledger) so re-running
    this infra cell — or any phase — never clobbers other gates' results."""
    gates = get_gates()
    gates[str(gate)] = {
        "passed": bool(passed),
        "detail": str(detail),
        "ts": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    _atomic_write_bytes(
        _GATE_FILE,
        json.dumps(gates, indent=2, default=str).encode("utf-8"),
    )
    status = "PASS" if passed else "FAIL"
    log(f"[gate {gate}] {status} — {detail}")
    return gates


# ----------------------------------------------------------------------------
# 8. MODEL-RELOAD GUARD PATTERN (note only — no GPU work here)
# ----------------------------------------------------------------------------
# The HookedTransformer 'model' and 'tokenizer' CANNOT be pickled across GPU
# disconnects, so they are NOT artifacts. Phase 0 reloads them, guarded like:
#
#     if "model" not in globals():
#         model, tokenizer = load_model_phase0(CFG)   # defined in Phase 0 cell
#
# Re-running Phase 0 after a reconnect rebuilds them in-memory; everything else
# (datasets, caches, gate ledger) is restored from ART via the helpers above.
# This cell intentionally does NONE of that — it only sets up the scaffolding.

# ----------------------------------------------------------------------------
# Record an environment snapshot + mark the infra gate as passed.
# ----------------------------------------------------------------------------
_env_snapshot = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "in_colab": _in_colab(),
    "has_torch": _HAS_TORCH,
    "torch_version": (torch.__version__ if _HAS_TORCH else None),
    "cuda_available": (bool(torch.cuda.is_available()) if _HAS_TORCH else False),
    "cuda_device": (torch.cuda.get_device_name(0)
                    if (_HAS_TORCH and torch.cuda.is_available()) else None),
    "numpy": np.__version__,
    "art": str(ART),
    "device": CFG["device"],
    "dtype": CFG["dtype_name"],
    "seed": CFG["seed"],
    "ts": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}
save_json("env_snapshot", _env_snapshot)

log(f"Infra ready. device={CFG['device']} dtype={CFG['dtype_name']} "
    f"seed={CFG['seed']} model={CFG['model_name']}")
log(f"Existing gates on disk: {list(get_gates().keys()) or '(none yet)'}")

# Self-check: prove the artifact round-trip works before any later cell trusts it.
# Use the kind-aware has_artifact so the existence check matches the loader.
_RT_KEY = "_infra_selfcheck"
save_json(_RT_KEY, {"ok": True, "seed": CFG["seed"]})
_rt_ok = has_artifact(_RT_KEY, "json") and load_json(_RT_KEY).get("ok") is True
set_gate("G_INFRA", _rt_ok,
         f"artifact round-trip + seeding OK; ART={ART}; device={CFG['device']}")
print("PASS: infra cell" if _rt_ok else "FAIL: infra cell artifact round-trip")

---
# Phase 0 — Environment & compute · **Gate G0**

Load and hook Llama‑3.1‑8B in bf16 on a single device, then run the five‑check smoke test:

1. model loads on the target device,
2. a forward pass on a sub‑30‑token arithmetic string returns finite logits,
3. `blocks.{L}.hook_resid_post` caches a `[batch, seq, 4096]` finite tensor,
4. `blocks.{L}.attn.hook_pattern` caches `[batch, n_heads, seq, seq]` with rows summing to ≈ 1,
5. greedy next‑token on `"12 + 7 ="` is a plausible digit.

**PASS** iff all five succeed → `G0` recorded. **FAIL → fallback:** HF `AutoModelForCausalLM` with manual `register_forward_hook` on the analogous modules. Decide by end of Day 0; don't iterate on library internals past that.


In [ ]:
# =====================================================================
# PHASE 0 — Gate G0 : load + hook Llama-3.1-8B, run 5-check smoke test
# Notebook contract: CFG, ART, has_artifact/save_*/load_*, log() already exist.
# 'model' / 'tokenizer' become globals after this cell.
#
# ADVERSARIAL-REVIEW FIXES vs original:
#   * set_gate() is NOT a contract-guaranteed helper -> call is now guarded
#     (NameError on the last line would otherwise fail G0 even on a clean pass).
#   * Smoke test now obeys the RESILIENCE RULE: if 'g0_smoke' already PASSED on
#     disk, we skip recomputation on reconnect instead of re-running the model.
#   * Fallback run_with_cache forward is hardened against output_attentions
#     being rejected by newer transformers (eager already fills attn_weights).
# transformer_lens API verified: HookedTransformer.from_pretrained(model_name,
#   hf_model=, tokenizer=, dtype=<torch.dtype>, device=) is correct; hook names
#   blocks.{L}.hook_resid_post / blocks.{L}.attn.hook_pattern /
#   blocks.{L}.hook_mlp_out are correct; Llama-3.1-8B -> n_layers=32, n_heads=32,
#   d_model=4096.
# =====================================================================
# ============================ PART A ================================
# HF AUTH + MODEL LOAD (guarded). Primary: transformer_lens HookedTransformer.
# Fallback: HF AutoModelForCausalLM wrapped to expose run_with_cache/run_with_hooks.
# ====================================================================
import os
import torch

# ---- HF auth: read token from env, never hardcode -------------------
# Llama-3.1-8B is a GATED repo. You must (a) have requested access on the HF
# model page with the same account, and (b) expose a token via env:
#     export HUGGINGFACE_TOKEN=hf_xxx   (or HF_TOKEN=hf_xxx)
_hf_token = os.environ.get("HUGGINGFACE_TOKEN") or os.environ.get("HF_TOKEN")
if _hf_token:
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=_hf_token, add_to_git_credential=False)
        log("HF: logged in from env token.")
    except Exception as e:
        log(f"HF: login() raised {type(e).__name__}: {e} (continuing; token may still be picked up from env).")
else:
    log("HF: no HUGGINGFACE_TOKEN / HF_TOKEN in env. "
        "If the gated Llama repo is inaccessible you will see a 401/403 at load; "
        "set the env var and re-run this cell.")

# ---- helper: confirm gated repo is reachable before the heavy load --
def _check_gated_repo(repo_id):
    """Return (ok: bool, msg: str). Cheap metadata probe; surfaces 401/403 clearly."""
    try:
        from huggingface_hub import model_info
        info = model_info(repo_id, token=_hf_token)
        return True, f"reachable (sha={getattr(info, 'sha', '?')})"
    except Exception as e:
        return False, (f"{type(e).__name__}: {e}\\n"
                       f"  -> Visit https://huggingface.co/{repo_id} and click 'Agree and access', "
                       f"then export HUGGINGFACE_TOKEN and re-run.")

_repo = CFG["model_name"]
_ok, _msg = _check_gated_repo(_repo)
log(f"HF: gated-repo check for {_repo}: {_msg}")

# ---- resolve the model revision/commit hash (for the version lock) --
def _resolve_revision(repo_id):
    try:
        from huggingface_hub import model_info
        return getattr(model_info(repo_id, token=_hf_token), "sha", None)
    except Exception:
        return None
_resolved_revision = _resolve_revision(_repo)

# ---- thin HF fallback wrapper ---------------------------------------
# Exposes the minimal surface later cells use:
#   .cfg.{n_layers,n_heads,d_model,device}
#   .to_tokens(str) -> LongTensor[1, seq]
#   .__call__(tokens) / .forward(tokens) -> logits[1, seq, vocab]
#   .run_with_cache(tokens) -> (logits, cache) where cache is dict keyed by
#       TL-style hook names: 'blocks.{L}.hook_resid_post',
#       'blocks.{L}.attn.hook_pattern', 'blocks.{L}.hook_mlp_out'
#   .run_with_hooks(tokens, fwd_hooks=[(name, fn), ...]) -> logits
# Attention pattern requires attn_implementation='eager' (no flash/sdpa).
# NOTE: in current transformers LlamaDecoderLayer.forward returns a *plain
# tensor* (resid_post) and LlamaMLP.forward returns a plain tensor, while
# LlamaAttention.forward returns (attn_output, attn_weights); the unwrap below
# ('out[0] if tuple else out') is robust across versions.
class _SimpleCache(dict):
    """dict that also accepts TL-ish indexing cache['blocks',L,'hook_resid_post']."""
    def __getitem__(self, key):
        if isinstance(key, tuple):
            # support cache['blocks', L, 'hook_resid_post'] -> 'blocks.{L}.hook_resid_post'
            parts = [str(k) for k in key]
            return dict.__getitem__(self, ".".join(parts))
        return dict.__getitem__(self, key)

class HFHookedWrapper:
    """Minimal transformer_lens-compatible shim over a HF CausalLM."""
    class _Cfg:
        pass

    def __init__(self, hf_model, hf_tokenizer, device):
        self._m = hf_model
        self.tokenizer = hf_tokenizer
        self.cfg = HFHookedWrapper._Cfg()
        cfg = hf_model.config
        self.cfg.n_layers = cfg.num_hidden_layers
        self.cfg.n_heads = cfg.num_attention_heads
        self.cfg.d_model = cfg.hidden_size
        self.cfg.d_vocab = cfg.vocab_size
        self.cfg.device = device
        self.device = device
        self._is_fallback = True

    def to_tokens(self, text, prepend_bos=True):
        enc = self.tokenizer(text, return_tensors="pt", add_special_tokens=prepend_bos)
        return enc["input_ids"].to(self.device)

    def to_str_tokens(self, text, prepend_bos=True):
        ids = self.to_tokens(text, prepend_bos=prepend_bos)[0]
        return [self.tokenizer.decode([i]) for i in ids.tolist()]

    @torch.no_grad()
    def forward(self, tokens, return_type="logits"):
        if isinstance(tokens, str):
            tokens = self.to_tokens(tokens)
        out = self._m(input_ids=tokens)
        return out.logits

    __call__ = forward

    @torch.no_grad()
    def run_with_cache(self, tokens, names_filter=None):
        """Returns (logits, cache). Caches resid_post / attn pattern / mlp_out
        per layer under TL-style names via forward hooks."""
        if isinstance(tokens, str):
            tokens = self.to_tokens(tokens)
        cache = _SimpleCache()
        handles = []

        def _keep(name):
            return (names_filter is None) or names_filter(name)

        layers = self._m.model.layers
        for L, block in enumerate(layers):
            rp_name = f"blocks.{L}.hook_resid_post"
            mlp_name = f"blocks.{L}.hook_mlp_out"
            attn_name = f"blocks.{L}.attn.hook_pattern"

            if _keep(rp_name):
                def _rp_hook(mod, inp, out, _n=rp_name):
                    h = out[0] if isinstance(out, tuple) else out
                    cache[_n] = h.detach()
                handles.append(block.register_forward_hook(_rp_hook))

            if _keep(mlp_name):
                def _mlp_hook(mod, inp, out, _n=mlp_name):
                    h = out[0] if isinstance(out, tuple) else out
                    cache[_n] = h.detach()
                handles.append(block.mlp.register_forward_hook(_mlp_hook))

            if _keep(attn_name):
                # eager attention returns attn weights as the 2nd output element
                def _attn_hook(mod, inp, out, _n=attn_name):
                    if isinstance(out, tuple) and len(out) >= 2 and out[1] is not None:
                        cache[_n] = out[1].detach()  # [b, n_heads, q, k]
                handles.append(block.self_attn.register_forward_hook(_attn_hook))

        try:
            # eager attn already fills attn_weights; output_attentions=True is
            # redundant and rejected on some newer transformers paths, so retry
            # without it if needed. The per-layer self_attn hooks fill the cache
            # either way.
            try:
                out = self._m(input_ids=tokens, output_attentions=True, use_cache=False)
            except (TypeError, ValueError) as _e_oa:
                log(f"HF fallback: output_attentions path rejected ({type(_e_oa).__name__}); "
                    f"retrying without it (eager hooks still capture patterns).")
                out = self._m(input_ids=tokens, use_cache=False)
            logits = out.logits
        finally:
            for h in handles:
                h.remove()
        return logits, cache

    @torch.no_grad()
    def run_with_hooks(self, tokens, fwd_hooks=None, return_type="logits"):
        """fwd_hooks: list of (tl_hook_name, fn(tensor, hook=None)->tensor|None).
        Supports resid_post / mlp_out (output rewrite). Attention-pattern editing
        is not supported in the fallback (rarely needed for G0)."""
        if isinstance(tokens, str):
            tokens = self.to_tokens(tokens)
        fwd_hooks = fwd_hooks or []
        handles = []
        layers = self._m.model.layers
        for name, fn in fwd_hooks:
            parts = name.split(".")
            L = int(parts[1])
            block = layers[L]
            if name.endswith("hook_resid_post"):
                target = block
            elif name.endswith("hook_mlp_out"):
                target = block.mlp
            else:
                raise NotImplementedError(f"fallback run_with_hooks does not support {name}")
            def _wrap(mod, inp, out, _fn=fn):
                h = out[0] if isinstance(out, tuple) else out
                new = _fn(h, hook=None)
                if new is None:
                    new = h
                if isinstance(out, tuple):
                    return (new,) + tuple(out[1:])
                return new
            handles.append(target.register_forward_hook(_wrap))
        try:
            logits = self._m(input_ids=tokens, use_cache=False).logits
        finally:
            for h in handles:
                h.remove()
        return logits

# ---- the guarded load -----------------------------------------------
USING_FALLBACK = False
if "model" not in globals():
    _device = CFG["device"]
    try:
        # ---- PRIMARY: transformer_lens HookedTransformer ----
        import transformer_lens
        from transformer_lens import HookedTransformer
        log("Loading via transformer_lens HookedTransformer (primary path)...")
        # Pass HF model+tokenizer explicitly so the gated download / dtype /
        # device placement is unambiguous and TL just wraps it. When hf_model is
        # supplied, TL skips its own dtype download path and uses these weights.
        from transformers import AutoModelForCausalLM, AutoTokenizer
        _hf_tok = AutoTokenizer.from_pretrained(CFG["model_name"], token=_hf_token)
        _hf_model = AutoModelForCausalLM.from_pretrained(
            CFG["model_name"],
            torch_dtype=torch.bfloat16,   # still accepted (deprecated alias of dtype)
            token=_hf_token,
        )
        model = HookedTransformer.from_pretrained(
            CFG["model_name"],
            hf_model=_hf_model,
            tokenizer=_hf_tok,
            dtype=torch.bfloat16,          # torch.dtype object is accepted
            device=_device,
            fold_ln=True,
            center_writing_weights=False,  # intentional: leave resid stream uncentered
            center_unembed=False,          # intentional for Llama analyses
        )
        tokenizer = model.tokenizer
        del _hf_model  # TL has copied the weights; free the HF copy
        log(f"Loaded HookedTransformer on {model.cfg.device} "
            f"(n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}, d_model={model.cfg.d_model}).")
    except Exception as e_primary:
        # ---- FALLBACK: raw HF + thin wrapper ----
        log(f"*** transformer_lens load FAILED: {type(e_primary).__name__}: {e_primary}")
        log("*** FALLING BACK to HF AutoModelForCausalLM + HFHookedWrapper. "
            "BUDGET NOTE: decide TL-vs-HF by end of Day 0; the fallback covers G0 "
            "but lacks turnkey activation/attribution patching for later phases.")
        from transformers import AutoModelForCausalLM, AutoTokenizer
        _hf_tok = AutoTokenizer.from_pretrained(CFG["model_name"], token=_hf_token)
        _hf_model = AutoModelForCausalLM.from_pretrained(
            CFG["model_name"],
            torch_dtype=torch.bfloat16,
            token=_hf_token,
            attn_implementation="eager",  # REQUIRED so self_attn returns attn weights
        ).to(_device)
        _hf_model.eval()
        model = HFHookedWrapper(_hf_model, _hf_tok, _device)
        tokenizer = model.tokenizer
        USING_FALLBACK = True
        log(f"Loaded HF fallback on {_device} "
            f"(n_layers={model.cfg.n_layers}, n_heads={model.cfg.n_heads}, d_model={model.cfg.d_model}).")
else:
    USING_FALLBACK = bool(getattr(model, "_is_fallback", False))
    log(f"'model' already in globals (fallback={USING_FALLBACK}); skipping reload.")

# ---- version lock (pin/log versions + resolved revision) ------------
def _ver(modname):
    try:
        return __import__(modname).__version__
    except Exception:
        return "n/a"
_versions = {
    "transformer_lens": _ver("transformer_lens"),
    "torch": torch.__version__,
    "transformers": _ver("transformers"),
    "huggingface_hub": _ver("huggingface_hub"),
    "model_name": CFG["model_name"],
    "resolved_revision": _resolved_revision,
    "using_fallback": USING_FALLBACK,
    "device": str(getattr(model.cfg, "device", CFG["device"])),
    "dtype": "bfloat16",
}
import json as _json
save_text("versions_lock", _json.dumps(_versions, indent=2))
log("versions_lock: " + _json.dumps(_versions))


# ============================ PART B ================================
# G0 SMOKE TEST — 5 checks. Each prints PASS/FAIL; register the gate at end.
# RESILIENCE: if a prior PASS is already on disk, skip recomputation.
# Uses EXACT transformer_lens hook-point names:
#   resid_post : f"blocks.{L}.hook_resid_post"   -> [batch, seq, d_model=4096]
#   attn patt. : f"blocks.{L}.attn.hook_pattern" -> [batch, n_heads, seq, seq]
# run_with_cache returns (logits, cache); index cache by the string hook name.
# ====================================================================
import torch

def _g0_smoke():
    results = {}
    PROMPT = "12 + 7 ="
    device_str = str(getattr(model.cfg, "device", CFG["device"]))
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    d_model = model.cfg.d_model
    L = min(5, n_layers - 1)  # arbitrary probe layer

    # --- Check 1: model loaded on target device ---
    try:
        on_target = (CFG["device"].split(":")[0] in device_str)
        results["1_device"] = bool(on_target)
        print(f"[G0.1] model on target device: device={device_str}, "
              f"target={CFG['device']} -> {'PASS' if on_target else 'FAIL'}")
    except Exception as e:
        results["1_device"] = False
        print(f"[G0.1] FAIL ({type(e).__name__}: {e})")

    # tokenize once, assert < 30 tokens
    tokens = model.to_tokens(PROMPT)
    seq = tokens.shape[1]
    assert seq < 30, f"smoke prompt unexpectedly long: {seq} tokens"

    # --- Check 2: forward pass -> finite logits ---
    try:
        with torch.no_grad():
            logits = model(tokens)
        finite = bool(torch.isfinite(logits).all().item())
        shape_ok = (logits.shape[0] == 1 and logits.shape[1] == seq)
        ok2 = finite and shape_ok
        results["2_forward_finite"] = ok2
        print(f"[G0.2] forward on {PROMPT!r}: logits {tuple(logits.shape)}, "
              f"all_finite={finite} -> {'PASS' if ok2 else 'FAIL'}")
    except Exception as e:
        results["2_forward_finite"] = False
        logits = None
        print(f"[G0.2] FAIL ({type(e).__name__}: {e})")

    # --- run_with_cache once for checks 3 & 4 ---
    rp_name = f"blocks.{L}.hook_resid_post"
    pat_name = f"blocks.{L}.attn.hook_pattern"
    try:
        with torch.no_grad():
            _logits_c, cache = model.run_with_cache(tokens)
    except Exception as e:
        cache = {}
        print(f"[G0.cache] run_with_cache FAILED ({type(e).__name__}: {e})")

    # --- Check 3: resid_post hook shape [batch, seq, 4096] + finite ---
    try:
        rp = cache[rp_name]
        shape_ok = (tuple(rp.shape) == (1, seq, d_model)) and (d_model == 4096)
        finite = bool(torch.isfinite(rp).all().item())
        ok3 = shape_ok and finite
        results["3_resid_post"] = ok3
        print(f"[G0.3] {rp_name}: shape={tuple(rp.shape)} "
              f"(expect (1,{seq},{d_model})), finite={finite} -> {'PASS' if ok3 else 'FAIL'}")
    except Exception as e:
        results["3_resid_post"] = False
        print(f"[G0.3] FAIL ({type(e).__name__}: {e})")

    # --- Check 4: attn pattern shape [batch, n_heads, seq, seq] + rows sum ~1 ---
    try:
        pat = cache[pat_name]
        shape_ok = (tuple(pat.shape) == (1, n_heads, seq, seq))
        row_sums = pat.float().sum(dim=-1)          # [1, n_heads, seq]
        close = torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-2)
        max_dev = float((row_sums - 1.0).abs().max().item())
        ok4 = shape_ok and close
        results["4_attn_pattern"] = ok4
        assert close, f"attn rows do not sum to 1 (max dev {max_dev:.3e})"
        print(f"[G0.4] {pat_name}: shape={tuple(pat.shape)} "
              f"(expect (1,{n_heads},{seq},{seq})), max|rowsum-1|={max_dev:.2e} "
              f"-> {'PASS' if ok4 else 'FAIL'}")
    except Exception as e:
        results["4_attn_pattern"] = False
        print(f"[G0.4] FAIL ({type(e).__name__}: {e})")

    # --- Check 5: greedy next-token decodes to a plausible digit ---
    try:
        if logits is None:
            with torch.no_grad():
                logits = model(tokens)
        next_id = int(logits[0, -1].argmax().item())
        next_tok = tokenizer.decode([next_id])
        stripped = next_tok.strip()
        ok5 = any(ch.isdigit() for ch in stripped) and len(stripped) > 0
        results["5_greedy_digit"] = ok5
        print(f"[G0.5] greedy next-token after {PROMPT!r}: "
              f"id={next_id} tok={next_tok!r} -> {'PASS' if ok5 else 'FAIL'} "
              f"(plausible digit; note correct answer is 19)")
    except Exception as e:
        results["5_greedy_digit"] = False
        print(f"[G0.5] FAIL ({type(e).__name__}: {e})")

    return results

# ---- RESILIENCE: reuse a prior PASS instead of recomputing on reconnect ----
if has_artifact("g0_smoke"):
    _g0_record = load_json("g0_smoke")
    if _g0_record.get("pass", False):
        _g0_results = _g0_record.get("checks", {})
        _g0_pass = True
        log("G0 smoke already PASSED on disk; skipping recompute (resilience).")
    else:
        _g0_results = _g0_smoke()
        _g0_pass = all(_g0_results.values()) if _g0_results else False
else:
    _g0_results = _g0_smoke()
    _g0_pass = all(_g0_results.values()) if _g0_results else False

print("\\n" + "=" * 56)
print(f"G0 SMOKE SUMMARY: {sum(bool(v) for v in _g0_results.values())}/5 checks passed "
      f"(fallback={USING_FALLBACK})")
for k, v in _g0_results.items():
    print(f"   {k:20s}: {'PASS' if v else 'FAIL'}")
print(f"GATE G0 -> {'PASS' if _g0_pass else 'FAIL'}")
print("=" * 56)

# persist the smoke result (light artifact) — this is the contract-guaranteed
# source of truth for the gate.
_g0_record = {
    "pass": bool(_g0_pass),
    "checks": {k: bool(v) for k, v in _g0_results.items()},
    "using_fallback": USING_FALLBACK,
    "versions": _versions,
}
save_json("g0_smoke", _g0_record)

# register the gate IFF the harness provides set_gate; it is NOT promised by the
# notebook contract, so guard it to avoid a NameError that would fail a clean G0.
if "set_gate" in globals() and callable(globals()["set_gate"]):
    set_gate("G0", bool(_g0_pass), _g0_record)
    log("G0 registered via set_gate().")
else:
    log("set_gate not available in this kernel; G0 result persisted to 'g0_smoke' artifact only.")

## Phase 1 — Gate G1: Novelty Check

**Controlled contribution (framing).** This project isolates whether Llama-3.1-8B's internal representation of an arithmetic expression encodes *structural nesting depth* as a quantity distinct from token content and sequence length. The core instrument is a **token-identical additive-identity depth contrast**: pairs of expressions that are *literally the same token sequence* up to insertion of additive-identity / parenthesization elements (e.g. `((a+0)+b)` vs `(a+(0+b))`, or `a+b` wrapped to varying parenthesis depth) so that the numeric value, the operand tokens, and (in the matched arm) the token count are held fixed while *only* the structural/parse depth varies. We pair this with (i) a **padding-invariance probe** — re-running the same contrast under length-/position-shifted padding to test whether any decoded "depth" signal is actually a positional or sequence-length artifact rather than a structural one; (ii) a **depth-2 composition** test that checks whether a depth signal recovered at depth 1 extends to nested composition; and (iii) a **causal probe check** that goes beyond decodability — patching/ablating the candidate depth direction and measuring behavioral effect, so a merely *decodable* depth direction is not mistaken for a *causally used* one. The combination (token-identical additive-identity contrast + padding-invariance + causal check, on a production 8B model) is the controlled unit we claim is new.

### Related work

| paper | contrast used | max depth | token-control? | distance factor? | causal check? |
|---|---|---|---|---|---|
| Hewitt & Manning 2019, *Structural Probe* (NAACL; aclanthology N19-1419) | tree **distance** & tree **depth** decoded from BERT/ELMo word reps (natural language) | full sentence parse depth | no (NL sentences, not token-matched) | yes (explicit L2-distance & L2-norm=depth probes) | no (decodability only) |
| Hewitt & Liang 2019 / probing-control line | probe vs control-task selectivity | n/a | partial (control tasks) | no | no |
| Stolfo, Belinkov & Sachan 2023, *Mechanistic Interpretation of Arithmetic Reasoning via Causal Mediation* (EMNLP; arXiv 2305.15054) | clean vs corrupted operands in arithmetic prompts | shallow (single op) | partial (operand swaps) | no (depth not a variable) | **yes** (causal mediation) |
| Hanna, Liu & Variengien 2023, *How does GPT-2 compute greater-than?* (NeurIPS; arXiv 2305.00586) | year-boundary `greater-than` circuit | single relation | partial | no | **yes** (circuit ablation) |
| Quirke & Barez 2023/24, *Understanding Addition in Transformers* (arXiv 2310.13121; ext. 2402.02619) | digit-wise addition algorithm in a 1-layer model | n/a (flat addition) | n/a | no | yes (ablation, trained toy model) |
| Nikankin, Reusch, Mueller & Belinkov 2024, *Arithmetic Without Algorithms: Bag of Heuristics* (arXiv 2410.21272; incl. Llama-3-8B) | operand-range / modulo / pattern heuristics via activation patching | flat arithmetic, no nesting | partial (operand-value sweeps) | no (structural depth not studied) | **yes** (activation patching, sparse-neuron circuit) |
| Murty et al. 2023, *Grokking of Hierarchical Structure in Vanilla Transformers* (ACL; aclanthology 2023.acl-short.38) | hierarchical generalization splits | tree-structured | no (behavioral) | partial | no |
| Yao, Peng et al. / Princeton 2021, *Self-Attention Networks Can Process Bounded Hierarchical Languages* (ACL; arXiv 2105.11115) — Dyck-(k,D) | balanced-bracket recognition | bounded nesting D | no (toy Dyck) | partial (depth as task variable) | no (construction + behavior) |
| Sharma, Dawes & Raval 2026, *Dissociating Decodability and Causal Use in Bracket-Sequence Transformers* (arXiv 2604.22128) — **nearest hit** | depth / distance / top-of-stack on Dyck transformers | bounded Dyck nesting | no (toy bracket seqs, not token-identical arithmetic) | **yes** (depth & distance decoded) | **yes** (attention-mask & residual-subspace ablation; finds depth decodable but not causally used) |
| AST-Probe (Sarda et al. 2022, ASE; dl.acm 3551349.3556900) | recover AST subspace from code-model hidden states | full AST | no (code, not matched) | yes (tree-recovery) | no (decodability) |
| Depth-generalization line 2025 (e.g. arXiv 2512.02677, 2510.02524) | recursion depth vs length in LM behavior | deep nested expr | no (behavioral accuracy) | partial (depth vs length) | no |
| **THIS project** | **token-identical additive-identity depth contrast** (value, operands, and token count held fixed; only parse depth varies) **+ padding-invariance probe** **+ depth-2 composition** | depth 1 → 2 (composition) | **yes — token-identical pairs by construction** | yes (depth isolated from length/position via padding control) | **yes — causal patch/ablation of the depth direction on Llama-3.1-8B** |

### G1 verdict

**PASS-repositionable.** No located paper runs the same controlled token-identical additive-identity depth contrast paired with a padding-invariance probe on the Llama-3 model class; the nearest hit (Sharma–Dawes–Raval 2026) performs an analogous *decodability-vs-causal-use* dissociation but on toy Dyck bracket sequences rather than token-matched additive-identity arithmetic, and Nikankin et al. 2024 use Llama-3-8B but study flat-arithmetic heuristics with no structural-depth variable — so the contribution is novel but should be framed explicitly against this decodability/causal-use line rather than as the first depth probe.

---
# Phase 2 — Stimulus construction & assertion harness · **Gate G2**

**The buildable‑today, validity‑critical artifact.** If G2 fails, no GPU result is trustworthy.

- **Factor A (Depth):** `( 0 + B ) * C` vs `0 + ( B * C )` — token‑identical, both evaluate to `B*C`, parens in both; only the structural boundary moves. Additive‑identity (`0 +`), **never** multiply‑by‑one (which the model may compile to a no‑op).
- **Factor B (Distance, the lead result):** same expression padded with `+ 0` chains that grow token count but preserve answer **and** tree depth.
- **Factor C (Depth‑2, the upside):** a second answer‑preserving nesting for the head‑reuse test later.

Every emitted pair must pass the machine‑checked assertions — **token‑length equality under the real tokenizer** (the hazard: multi‑digit numbers tokenize inconsistently), operand‑position equality, final‑answer equality, parens‑present, and the depth‑tree (differs for A, equal for B). Violations are **dropped with a logged reason**.


In [ ]:
# ============================================================================
# Phase 2 — Gate G2 : controlled stimulus generator + machine-checked assertion
#                     harness.  THE single most validity-critical artifact.
# ----------------------------------------------------------------------------
# Pure CPU. Uses ONLY the model TOKENIZER (no forward pass), so it runs even
# before the GPU is attached. Resumable: the whole dataset is guarded by
# has_artifact('dataset_phase2','pickle'); the Phase-3-facing JSON view is
# guarded by has_artifact('phase2_stimuli','json').
#
# DESIGN (spec-faithful, parenthesized additive-identity contrast):
#   Factor A (Depth):   depth_left  = "( 0 + B ) * C ="   -> (0+B)*C = B*C
#                       depth_right = "0 + ( B * C ) ="   -> 0+(B*C) = B*C
#     Same token multiset {(, ), 0, +, *, =, B, C}; both evaluate to B*C;
#     parentheses present in BOTH; only the paren BOUNDARY moves. Additive
#     identity (0+) -- NEVER multiply-by-one, which the model may compile to a
#     no-op -- so the multiplication operands B,C stay genuinely engaged.
#   Factor B (Distance, the lead result): SUFFIX padding " + 0" * k inserted
#     before "=". Grows token count, preserves answer AND the *-nesting depth.
#     The probed operand's distance to the final operator/'=' shifts by +2k;
#     we RECORD the shift (spec: track, don't forbid).
#   Factor C (Depth-2, the upside): "( 0 + ( 0 + B ) * C ) * D =" = B*C*D,
#     every bracket additive-identity-guarded. Generated+validated now, used
#     in Phase 8.
#
# MACHINE-CHECKED ASSERTIONS (tokenized with the REAL model tokenizer; compare
# TOKEN-ID lengths, never char/whitespace lengths):
#   Factor A pair (depth_left vs depth_right, same B,C):
#     [HARD] token_len(left)        == token_len(right)
#     [HARD] B_token_index(left)    == B_token_index(right)     (held fixed)
#     [HARD] answer(left)           == answer(right)            (Python ground truth)
#     [HARD] parens_present(left) and parens_present(right)
#     [HARD] tree_depth(left)       != tree_depth(right)        (the boundary moves)
#     [REC ] C_token_index shift recorded as metadata (structurally must move).
#   Factor B (pad_0 vs pad_k, same expr):
#     [HARD] answer(pad_0)          == answer(pad_k)
#     [HARD] tree_depth(pad_0)      == tree_depth(pad_k)         (padding != depth)
#     [HARD] token_len(pad_k)        > token_len(pad_0)          (it really grew)
#     [REC ] probed-operand distance-to-final-operator shift recorded.
# Violations DROP the pair with a logged reason; the assertion_report counts
# drops per factor and per reason so a tokenizer fighting the design is visible.
# ============================================================================

import re
import itertools
import numpy as np

assert "tokenizer" in globals(), "Phase 2 needs `tokenizer` (run Phase 0 first)."

# ----------------------------------------------------------------------------
# 0) Parameters (recorded in CFG so the band/volume are auditable artifacts).
# ----------------------------------------------------------------------------
CFG.setdefault("g2_target_per_factor", 2000)      # aim: low thousands of clean pairs.
CFG.setdefault("g2_min_valid_per_factor", 800)    # G2 PASS floor per factor.
CFG.setdefault("g2_sample_budget", 60000)         # max (B,C) draws before giving up.
# Digit-band grid the generator sweeps so Phase 3 can locate the must-compute band.
# (b_digits, c_digits) inclusive digit counts. Two-digit x {one,two,three}-digit is
# the spec's primary "not-a-memorized-product" zone; we span around it.
CFG.setdefault("g2_digit_grid", [[1, 2], [2, 1], [2, 2], [2, 3], [3, 2], [3, 3], [1, 3], [3, 1]])
CFG.setdefault("g2_pad_lengths", list(CFG.get("pad_lengths", [0, 2, 4, 8, 16])))
SEP = " "                 # single space between every surface token.
ANSWER_CUE = "="          # answer cue; model predicts the next token after it.
PAD_UNIT = ["+", "0"]     # one suffix padding identity op:  ... + 0 ...
_seed = int(CFG.get("seed", 0))

# ----------------------------------------------------------------------------
# 1) BOS offset (authoritative for Phase 4). How many special tokens the
#    tokenizer prepends under add_special_tokens=True.
# ----------------------------------------------------------------------------
def _compute_bos_offset():
    with_sp = tokenizer("0", add_special_tokens=True)["input_ids"]
    no_sp   = tokenizer("0", add_special_tokens=False)["input_ids"]
    return max(0, len(with_sp) - len(no_sp))

BOS_OFFSET = _compute_bos_offset()
CFG["bos_offset"] = BOS_OFFSET
save_json("phase2_bos_offset", BOS_OFFSET)
log(f"Phase 2: BOS offset = {BOS_OFFSET} (special tokens prepended).")

# Does the tokenizer expose char offset mapping? (Fast tokenizers do; Llama-3.1 is fast.)
def _supports_offsets():
    try:
        enc = tokenizer("0 + 1", return_offsets_mapping=True, add_special_tokens=True)
        return "offset_mapping" in enc
    except Exception:
        return False

_HAS_OFFSETS = _supports_offsets()
log(f"Phase 2: tokenizer offset_mapping available = {_HAS_OFFSETS}")

# ----------------------------------------------------------------------------
# 2) Surface rendering. We build the surface as an ordered list of (text, role)
#    SEGMENTS so we know each operand's exact char span -> exact token span,
#    robust to multi-digit operands splitting into >1 token.
#    role in {'lparen','rparen','op0','plus','star','B','C','D','pad_plus',
#             'pad_zero','eq'}.
# ----------------------------------------------------------------------------
def _segments(template, B, C, pad_len=0, D=None):
    """Return ordered list of (text, role) segments for a stimulus surface."""
    B, C = str(int(B)), str(int(C))
    if template == "depth_left":          # ( 0 + B ) * C
        segs = [("(", "lparen"), ("0", "op0"), ("+", "plus"), (B, "B"),
                (")", "rparen"), ("*", "star"), (C, "C")]
    elif template == "depth_right":       # 0 + ( B * C )
        segs = [("0", "op0"), ("+", "plus"), ("(", "lparen"), (B, "B"),
                ("*", "star"), (C, "C"), (")", "rparen")]
    elif template == "depth2":            # ( 0 + ( 0 + B ) * C ) * D  = B*C*D
        D = str(int(D))
        segs = [("(", "lparen"), ("0", "op0"), ("+", "plus"),
                ("(", "lparen"), ("0", "op0"), ("+", "plus"), (B, "B"), (")", "rparen"),
                ("*", "star"), (C, "C"), (")", "rparen"), ("*", "star"), (D, "D")]
    else:
        raise ValueError(f"unknown template {template!r}")
    # SUFFIX padding: append k copies of " + 0" before the answer cue.
    for _ in range(int(pad_len)):
        segs += [("+", "pad_plus"), ("0", "pad_zero")]
    segs += [(ANSWER_CUE, "eq")]
    return segs

def _assemble(segs):
    """Join segments with SEP; return (text, list-of-(start,end,role) char spans)."""
    parts, spans, pos = [], [], 0
    for i, (txt, role) in enumerate(segs):
        if i > 0:
            pos += len(SEP)
        start = pos
        end = pos + len(txt)
        spans.append((start, end, role))
        parts.append(txt)
        pos = end
    return SEP.join(parts), spans

# ----------------------------------------------------------------------------
# 3) Tokenize a surface and resolve each operand/operator token index.
#    Returns dict: token_ids, token_len, and role_index = {role: first_tok_idx}
#    (for repeated roles like 'star'/'lparen' we keep a list).
# ----------------------------------------------------------------------------
def _locate_tokens(text, spans):
    if _HAS_OFFSETS:
        enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=True)
        ids = enc["input_ids"]
        offs = enc["offset_mapping"]
    else:
        enc = tokenizer(text, add_special_tokens=True)
        ids = enc["input_ids"]
        offs = _fallback_offsets(text, ids)
    role_index = {}
    for (cs, ce, role) in spans:
        idx = None
        for ti, (s, e) in enumerate(offs):
            if e <= s:                       # special token / empty span -> skip
                continue
            if s >= cs and e <= ce:          # token fully inside the segment span
                idx = ti
                break
            if s < ce and e > cs and idx is None:  # overlap fallback (rare BPE merge)
                idx = ti
        role_index.setdefault(role, []).append(idx)
    return ids, role_index

def _fallback_offsets(text, ids):
    """Approximate per-token char spans for a slow tokenizer by decoding tokens
    and walking the string. Best-effort; only used if offset_mapping is absent."""
    offs, cursor = [], 0
    for tid in ids:
        piece = tokenizer.decode([tid])
        stripped = piece.strip()
        if stripped == "":
            offs.append((0, 0)); continue
        j = text.find(stripped, cursor)
        if j < 0:
            offs.append((0, 0))
        else:
            offs.append((j, j + len(stripped))); cursor = j + len(stripped)
    return offs

# ----------------------------------------------------------------------------
# 4) Ground-truth evaluation and structural tree depth.
#    tree_depth here = paren-NESTING depth of the multiplication operator '*'
#    (the operation under test): depth_left -> 0, depth_right -> 1. SUFFIX
#    padding does not enclose '*', so it leaves this invariant unchanged --
#    which is exactly what the Factor B assertion requires.
# ----------------------------------------------------------------------------
def _eval_answer(template, B, C, D=None):
    if template == "depth_left":   return (0 + int(B)) * int(C)
    if template == "depth_right":  return 0 + (int(B) * int(C))
    if template == "depth2":       return (0 + (0 + int(B)) * int(C)) * int(D)
    raise ValueError(template)

def _star_nesting_depth(template):
    """Paren-nesting depth of the (primary) '*' operator."""
    if template == "depth_left":   return 0     # '(0+B) * C' : '*' outside all parens
    if template == "depth_right":  return 1     # '0 + (B*C)' : '*' one paren deep
    if template == "depth2":       return 2     # deepest '*' two parens deep
    raise ValueError(template)

def _parens_present(text):
    return ("(" in text) and (")" in text)

# ----------------------------------------------------------------------------
# 5) Record builder. One record == one fully-described surface.
# ----------------------------------------------------------------------------
def make_record(template, B, C, factor, pad_len=0, D=None):
    segs = _segments(template, B, C, pad_len=pad_len, D=D)
    text, spans = _assemble(segs)
    ids, role_index = _locate_tokens(text, spans)
    ans = _eval_answer(template, B, C, D=D)
    op_idx = {
        "B": role_index.get("B", [None])[0],
        "C": role_index.get("C", [None])[0],
    }
    if D is not None:
        op_idx["D"] = role_index.get("D", [None])[0]
    star_list = role_index.get("star", [None])
    rec = {
        "prompt": text,                 # Phase 3 reads this field.
        "expr_string": text,
        "factor": factor,               # 'A' | 'B' | 'C'
        "condition": template,          # depth_left | depth_right | depth2
        "B": int(B), "C": int(C),
        "answer": int(ans),
        "pad_len": int(pad_len),
        "token_ids": [int(t) for t in ids],
        "token_len": len(ids),
        "operand_token_indices": op_idx,
        "operator_token_index": (None if star_list[0] is None else int(star_list[0])),
        "op0_token_index": role_index.get("op0", [None])[0],
        "eq_token_index": role_index.get("eq", [None])[0],
        "tree_depth": _star_nesting_depth(template),
        "parens": _parens_present(text),
    }
    if D is not None:
        rec["D"] = int(D)
    return rec

# ----------------------------------------------------------------------------
# 6) Assertion harnesses. Each returns (ok: bool, reason: str|None).
# ----------------------------------------------------------------------------
def assert_factorA(recL, recR):
    if recL["token_len"] != recR["token_len"]:
        return False, "token_length_mismatch"
    bi_L = recL["operand_token_indices"]["B"]; bi_R = recR["operand_token_indices"]["B"]
    if bi_L is None or bi_R is None:
        return False, "B_not_located"
    if bi_L != bi_R:
        return False, "B_position_mismatch"
    if recL["operand_token_indices"]["C"] is None or recR["operand_token_indices"]["C"] is None:
        return False, "C_not_located"
    if recL["answer"] != recR["answer"]:
        return False, "answer_mismatch"
    if not (recL["parens"] and recR["parens"]):
        return False, "parens_absent"
    if recL["tree_depth"] == recR["tree_depth"]:
        return False, "tree_depth_not_differing"
    if recL["operator_token_index"] is None or recR["operator_token_index"] is None:
        return False, "star_not_located"
    return True, None

def assert_factorB(rec0, reck):
    if rec0["answer"] != reck["answer"]:
        return False, "answer_mismatch"
    if rec0["tree_depth"] != reck["tree_depth"]:
        return False, "tree_depth_changed_by_padding"
    if not (reck["token_len"] > rec0["token_len"]):
        return False, "token_length_did_not_grow"
    if reck["operand_token_indices"]["C"] is None:
        return False, "C_not_located"
    return True, None

# ----------------------------------------------------------------------------
# 7) Operand sampling across the digit grid (de-duplicated, non-trivial).
# ----------------------------------------------------------------------------
def _draw_operand(rng, ndig):
    lo = 10 ** (ndig - 1) if ndig > 1 else 2      # avoid 0/1 (×0,×1 are trivial no-ops)
    hi = (10 ** ndig) - 1
    return int(rng.integers(lo, hi + 1))

def _nontrivial(B, C):
    # exclude memorized-ish: single-digit×single-digit, and exact powers of ten products.
    if B < 2 or C < 2:
        return False
    prod = B * C
    if B <= 9 and C <= 9:
        return False
    return True

# ----------------------------------------------------------------------------
# 8) GENERATE (guarded). Build Factor A pairs, Factor B padding series, Factor C.
# ----------------------------------------------------------------------------
def _build_dataset():
    rng = np.random.default_rng(_seed)
    grid = [tuple(g) for g in CFG["g2_digit_grid"]]
    target = int(CFG["g2_target_per_factor"])
    budget = int(CFG["g2_sample_budget"])
    pads = sorted(set(int(k) for k in CFG["g2_pad_lengths"] if int(k) > 0))

    factorA, factorB, factorC = [], [], []
    drops = {"A": {}, "B": {}, "C": {}}
    seen = set()

    def _drop(factor, reason):
        drops[factor][reason] = drops[factor].get(reason, 0) + 1

    draws = 0
    gi = 0
    while len(factorA) < target and draws < budget:
        bdig, cdig = grid[gi % len(grid)]; gi += 1
        B = _draw_operand(rng, bdig); C = _draw_operand(rng, cdig)
        draws += 1
        if not _nontrivial(B, C):
            _drop("A", "trivial_operand"); continue
        key = (B, C)
        if key in seen:
            continue
        seen.add(key)

        recL = make_record("depth_left", B, C, factor="A")
        recR = make_record("depth_right", B, C, factor="A")
        okA, why = assert_factorA(recL, recR)
        if not okA:
            _drop("A", why); continue
        # Record C's structural shift (tracked, not forbidden).
        c_shift = recL["operand_token_indices"]["C"] - recR["operand_token_indices"]["C"]
        recL["C_index_shift_vs_pair"] = int(c_shift)
        recR["C_index_shift_vs_pair"] = int(-c_shift)
        pair_id = f"A_{B}x{C}"
        recL["pair_id"] = pair_id; recR["pair_id"] = pair_id
        factorA.append(recL); factorA.append(recR)

        # ---- Factor B: padding series on the depth_right base for this (B,C). ----
        base = make_record("depth_right", B, C, factor="B", pad_len=0)
        series_ok = True
        padded = []
        for k in pads:
            reck = make_record("depth_right", B, C, factor="B", pad_len=k)
            okB, whyB = assert_factorB(base, reck)
            if not okB:
                _drop("B", whyB); series_ok = False; break
            # distance from probed operand C to the final operator '=' (grows with k).
            reck["C_distance_to_eq"] = int(reck["eq_token_index"] - reck["operand_token_indices"]["C"])
            padded.append(reck)
        if series_ok and padded:
            base["C_distance_to_eq"] = int(base["eq_token_index"] - base["operand_token_indices"]["C"])
            base["pad_series_id"] = pair_id
            for r in padded:
                r["pad_series_id"] = pair_id
            factorB.append(base); factorB.extend(padded)

        # ---- Factor C: depth-2, answer-preserving (validated, used in Phase 8). ----
        if len(factorC) < target:
            D = _draw_operand(rng, max(1, bdig))
            if D >= 2:
                recC = make_record("depth2", B, C, factor="C", D=D)
                if recC["answer"] == B * C * D and recC["parens"] \
                        and recC["operator_token_index"] is not None:
                    factorC.append(recC)
                else:
                    _drop("C", "depth2_validation_failed")

    return {
        "factorA": factorA, "factorB": factorB, "factorC": factorC,
        "drops": drops, "draws": draws,
        "surface_spec": {
            "templates": ["depth_left", "depth_right", "depth2"],
            "separator": SEP, "answer_cue": ANSWER_CUE, "pad_unit": PAD_UNIT,
            "pad_style": "suffix_before_eq", "bos_offset": BOS_OFFSET,
        },
    }

if has_artifact("dataset_phase2", "pickle"):
    DATA = load_pickle("dataset_phase2")
    log(f"Phase 2: loaded cached dataset (A={len(DATA['factorA'])}, "
        f"B={len(DATA['factorB'])}, C={len(DATA['factorC'])}).")
else:
    log("Phase 2: generating controlled stimuli (CPU; tokenizer only)...")
    DATA = _build_dataset()
    save_pickle("dataset_phase2", DATA)
    log("Phase 2: dataset generated and cached.")

# ----------------------------------------------------------------------------
# 9) Phase-3-facing JSON view: the Factor A experimental stimuli, both
#    conditions, with {prompt, B, C, answer, ...}. Saved under 'phase2_stimuli'
#    (the first name Phase 3 searches).
# ----------------------------------------------------------------------------
if not has_artifact("phase2_stimuli", "json"):
    save_json("phase2_stimuli", DATA["factorA"])
    save_json("phase2_surface_spec", DATA["surface_spec"])
    log(f"Phase 2: wrote phase2_stimuli (n={len(DATA['factorA'])}) for Phase 3.")

# ----------------------------------------------------------------------------
# 10) Assertion report + G2 gate.
# ----------------------------------------------------------------------------
nA_pairs = len(DATA["factorA"]) // 2
nB_series = sum(1 for r in DATA["factorB"] if r.get("pad_len", 1) == 0)
nC = len(DATA["factorC"])
floor = int(CFG["g2_min_valid_per_factor"])

def _fmt_drops(d):
    return ", ".join(f"{k}={v}" for k, v in sorted(d.items())) or "(none)"

report = []
report.append("==================== PHASE 2 / GATE G2 ASSERTION REPORT ====================")
report.append(f"sampling draws used        : {DATA['draws']} / budget {CFG['g2_sample_budget']}")
report.append(f"Factor A  valid pairs      : {nA_pairs}   (records={len(DATA['factorA'])})")
report.append(f"Factor A  drops            : {_fmt_drops(DATA['drops']['A'])}")
report.append(f"Factor B  valid pad-series : {nB_series} (records={len(DATA['factorB'])}, pads={CFG['g2_pad_lengths']})")
report.append(f"Factor B  drops            : {_fmt_drops(DATA['drops']['B'])}")
report.append(f"Factor C  valid depth-2    : {nC}")
report.append(f"Factor C  drops            : {_fmt_drops(DATA['drops']['C'])}")
report.append(f"BOS offset                 : {BOS_OFFSET}")
report.append(f"PASS floor per factor      : {floor}")

# Token-length parity sanity: every Factor A pair must be token-length-equal (it
# is, by construction of the drop rule) -- restate as an explicit invariant count.
parity_ok = all(
    DATA["factorA"][i]["token_len"] == DATA["factorA"][i + 1]["token_len"]
    for i in range(0, len(DATA["factorA"]), 2)
)
report.append(f"Factor A token-length parity (all pairs equal) : {parity_ok}")

# 20 random spot-reads for manual inspection (the spec's manual check).
rng = np.random.default_rng(_seed + 99)
report.append("--------------------------- 20 random spot-reads ---------------------------")
idxs = rng.choice(len(DATA["factorA"]), size=min(20, len(DATA["factorA"])), replace=False)
for j in idxs:
    r = DATA["factorA"][int(j)]
    report.append(f"  [{r['condition']:>11}] {r['prompt']:<22} = {r['answer']:<7} "
                  f"tok_len={r['token_len']} Bidx={r['operand_token_indices']['B']} "
                  f"Cidx={r['operand_token_indices']['C']} *idx={r['operator_token_index']} "
                  f"depth(*)={r['tree_depth']}")
report.append("===========================================================================")
report_text = "\n".join(report)
save_text("assertion_report", report_text)
print(report_text)

g2_pass = bool(parity_ok and nA_pairs >= floor and nB_series >= floor and nC >= floor)
g2_detail = (f"A_pairs={nA_pairs}, B_series={nB_series}, C={nC} (floor={floor}); "
             f"parity={parity_ok}; A_drops={_fmt_drops(DATA['drops']['A'])}")
set_gate("G2", g2_pass, g2_detail)

print(f"\nGATE G2: {'PASS' if g2_pass else 'FAIL'}  ({g2_detail})")
if not g2_pass:
    print("FAIL GUIDANCE:")
    if not parity_ok:
        print(" - Token-length parity broke for some pair -> the tokenizer is fighting the")
        print("   design. Inspect drops['A']['token_length_mismatch']; restrict g2_digit_grid")
        print("   to operand digit-counts that tokenize consistently, then re-run.")
    if nA_pairs < floor:
        print(f" - Only {nA_pairs} clean Factor-A pairs (< {floor}). Raise g2_sample_budget or")
        print("   widen g2_digit_grid to ranges that survive the token-length control.")
    if nB_series < floor:
        print(" - Too few padding series survived; check drops['B'] for the dominant reason.")
    if nC < floor:
        print(" - Too few depth-2 stimuli; relax Factor C operand draw or raise budget.")


---
# Phase 3 — Behavioral validation · **Gate G3** (first real kill‑switch)

Forward‑pass only, before any patching. Three checks, all cached to disk:

1. **Accuracy** — greedy decode on `(0+B)*C`‑class expressions; PASS if accuracy in the chosen band clears a recorded floor (default ≥ 80%).
2. **No‑op check** — vary `B`,`C`; the prediction must track `B*C` (the `0 +` must not let the model skip the multiplication). Guards the additive‑identity correction.
3. **Must‑compute check** — accuracy vs operand size must show *graceful degradation* (computation), **not** pinned‑100% (memorized lookup) and **not** chance.

Locks the operand band that Phases 4–9 consume. **FAIL → stop and redesign the stimulus** — this is the cheapest place to catch a fatal artifact.


In [ ]:
# Phase 3 — Gate G3 (first real kill-switch): behavioral validation, forward-pass only.
# Proves the model COMPUTES (not looks up) the operator-precedence stimulus and engages
# the operand IN-BAND, before any expensive probing. Three checks:
#   (1) ACCURACY        — greedy-decode (0+B)*C, robustly parse the int, vs ground truth.
#   (2) NO-OP CHECK      — predictions TRACK B*C as B,C vary; "0 +" doesn't kill the mult.
#   (3) MUST-COMPUTE     — accuracy vs operand size: graceful DEGRADATION (compute), not
#                          pinned-at-100% (lookup), not flat, not collapsed. LOCK that band.
#
# RESILIENCE: every forward pass is batched and guarded by has_artifact(...). A GPU
# disconnect mid-run resumes from cached eval results; re-running top-to-bottom recomputes
# nothing already on disk. Relies ONLY on model/tokenizer/CFG/ART/helpers from earlier cells.
#
# ADVERSARIAL-REVIEW FIXES vs prior draft:
#   * Padding-side bug: last real token is NOT mask.sum()-1 under left padding. We now
#     standardize tokenizer.padding_side='left' and read the TRUE last column, so the token
#     scored is exactly the token after "equals " regardless of pad side.
#   * Must-compute: 'graceful degradation' now requires a real accuracy DROP across the band,
#     so a flat (e.g. 50%) curve can no longer masquerade as computation.

import re, time
import numpy as np
import torch
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------------------------
# 0) Knobs (recorded to CFG so the floor/band are auditable artifacts, not magic numbers).
# ----------------------------------------------------------------------------------------
CFG.setdefault("g3_accuracy_floor", 0.80)      # PASS band must clear this overall accuracy.
CFG.setdefault("g3_eval_batch_size", 64)        # forward-pass batch (cheap GPU step).
CFG.setdefault("g3_max_answer_tokens", 6)       # greedy-decode budget for the answer int.
CFG.setdefault("g3_noop_grid", 24)              # B,C values per axis for the no-op sweep.
CFG.setdefault("g3_per_bin_n", 96)              # stimuli sampled per operand-size bin.
# Operand-size bins (max operand magnitude). Phase 4-9 consume the LOCKED subset of these.
CFG.setdefault("g3_operand_bins", [(2, 9), (10, 19), (20, 49), (50, 99), (100, 199), (200, 499)])
# "graceful degradation" band acceptance window (per-bin accuracy must sit inside this):
CFG.setdefault("g3_band_lo", 0.30)              # below this -> collapsed to chance (useless).
CFG.setdefault("g3_band_hi", 0.985)             # at/above this -> looks like memorized lookup.
CFG.setdefault("g3_min_band_drop", 0.15)        # locked band must DROP by >= this end-to-end
                                                #   (flat curve != computation -> reject).
ACC_FLOOR = float(CFG["g3_accuracy_floor"])
MIN_DROP  = float(CFG["g3_min_band_drop"])
seed = int(CFG.get("seed", 0))

# ----------------------------------------------------------------------------------------
# 1) set_gate fallback. Earlier cells normally define set_gate/GATES; tolerate their absence
#    so this cell is self-contained on a fresh kernel that only restored model/tokenizer.
# ----------------------------------------------------------------------------------------
if "set_gate" not in globals():
    def set_gate(name, passed, detail=""):
        gates = load_json("gates") if has_artifact("gates") else {}
        gates[name] = {"pass": bool(passed), "detail": str(detail), "ts": time.time()}
        save_json("gates", gates)
        log(f"[gate] {name} = {'PASS' if passed else 'FAIL'} :: {detail}")
        return gates[name]

# ----------------------------------------------------------------------------------------
# 2) Consume the Phase 2 stimuli artifact (tolerant: name + schema may vary across builds),
#    falling back to a *token-controlled* regenerator so G3 is runnable even if Phase 2's
#    exact field names differ. A stimulus is a dict with at least: prompt, B, C, answer.
# ----------------------------------------------------------------------------------------
def _coerce_stimuli(obj):
    """Normalize whatever Phase 2 saved into a list[dict(prompt,B,C,answer)]."""
    rows = obj
    if isinstance(obj, dict):
        for k in ("stimuli", "items", "data", "examples", "rows", "experimental", "exp"):
            if k in obj and isinstance(obj[k], (list, tuple)):
                rows = obj[k]; break
    out = []
    for r in rows:
        if not isinstance(r, dict):
            continue
        prompt = r.get("prompt", r.get("text", r.get("input", r.get("expr"))))
        B = r.get("B", r.get("b"))
        C = r.get("C", r.get("c"))
        ans = r.get("answer", r.get("target", r.get("gt", r.get("label", r.get("y")))))
        if prompt is None:
            continue
        if ans is None and B is not None and C is not None:
            ans = int(B) * int(C)
        if ans is None:
            continue
        out.append({"prompt": str(prompt),
                    "B": (None if B is None else int(B)),
                    "C": (None if C is None else int(C)),
                    "answer": int(ans)})
    return out

def _fmt_expr(B, C):
    # The additive-identity (0+B)*C stimulus. Phase 2 owns the canonical surface form; this
    # mirror is only used by the guarded fallback when no Phase 2 artifact is on disk.
    return f"{(0)}+{B} times {C} equals "  # word 'times' avoids '*' tokenization quirks.

def _load_phase2_stimuli():
    for nm in ("phase2_stimuli", "stimuli", "phase2_stims", "p2_stimuli",
               "stimuli_experimental", "phase2_exp", "phase2"):
        if has_artifact(nm):
            stim = _coerce_stimuli(load_json(nm))
            if stim:
                log(f"Phase 2 stimuli loaded from artifact '{nm}': n={len(stim)}")
                return stim, nm
    # --- guarded fallback: regenerate a token-controlled bank (cached) ---
    log("WARNING: no Phase 2 stimuli artifact found; regenerating a controlled fallback bank.")
    rng = np.random.default_rng(seed)
    stim = []
    for (lo, hi) in CFG["g3_operand_bins"]:
        for _ in range(CFG["g3_per_bin_n"]):
            B = int(rng.integers(lo, hi + 1)); C = int(rng.integers(lo, hi + 1))
            stim.append({"prompt": _fmt_expr(B, C), "B": B, "C": C, "answer": B * C})
    save_json("phase3_stimuli_fallback", stim)
    return stim, "phase3_stimuli_fallback"

if has_artifact("phase3_stimuli_resolved"):
    STIM = load_json("phase3_stimuli_resolved")
    src_name = load_text("phase3_stimuli_src") if has_artifact("phase3_stimuli_src") else "cached"
else:
    STIM, src_name = _load_phase2_stimuli()
    save_json("phase3_stimuli_resolved", STIM)
    save_text("phase3_stimuli_src", src_name)
log(f"G3 operating on {len(STIM)} stimuli (source='{src_name}'); accuracy floor={ACC_FLOOR}")

# ----------------------------------------------------------------------------------------
# 3) Robust greedy decode + integer parser. Works for transformer_lens HookedTransformer
#    (model(tokens) -> logits [B,T,V]) AND an HF-style fallback wrapper (-> .logits).
#    Batched, deterministic, no sampling. Parses multi-token numbers / leading-space tokens.
#
#    PADDING SAFETY (key fix): we force LEFT padding. With left padding the real tokens of
#    every row end at the final column, so the token to score (the one after "equals ") is
#    ALWAYS at index T-1 -- no mask-sum arithmetic, no left/right ambiguity. Newly generated
#    tokens are appended on the right, so after k steps the just-produced token is again the
#    last column. We still gather per-row by the true last attended index to be airtight even
#    if a model/tokenizer ignores our padding_side request.
# ----------------------------------------------------------------------------------------
_DEVICE = CFG.get("device", "cuda")

# Force a deterministic, generation-correct padding side once.
try:
    tokenizer.padding_side = "left"
except Exception as _e:
    log(f"(could not set tokenizer.padding_side='left': {_e})")

def _to_logits(out):
    return out.logits if hasattr(out, "logits") else out

@torch.no_grad()
def _encode(prompts):
    enc = tokenizer(list(prompts), return_tensors="pt", padding=True)
    ids = enc["input_ids"].to(_DEVICE)
    mask = enc.get("attention_mask")
    mask = None if mask is None else mask.to(_DEVICE)
    return ids, mask

@torch.no_grad()
def _safe_forward(ids, mask):
    """Forward that tolerates models whose __call__ doesn't accept attention_mask."""
    try:
        return model(ids, attention_mask=mask)
    except TypeError:
        return model(ids)

def _last_real_index(mask):
    """True index of the last attended (real) token per row, correct for BOTH pad sides.
    We take the position of the last '1' in the attention mask: argmax over reversed mask."""
    T = mask.shape[1]
    # index of last nonzero per row = T-1 - (#trailing zeros). Use flip+argmax of the
    # boolean mask to find the first real token from the right.
    flipped = torch.flip(mask, dims=[1])
    # argmax returns the FIRST max (first real token scanning from the right end)
    first_from_right = torch.argmax((flipped > 0).int(), dim=1)
    return (T - 1) - first_from_right

@torch.no_grad()
def _greedy_continuations(prompts, max_new=None):
    """Greedy-decode `max_new` tokens per prompt. Returns list[str] of generated text only
    (prompt stripped). Padding-side agnostic: score each row at its TRUE last real index."""
    max_new = max_new or CFG["g3_max_answer_tokens"]
    pad_id = tokenizer.pad_token_id
    if pad_id is None:
        pad_id = tokenizer.eos_token_id
        try: tokenizer.pad_token = tokenizer.eos_token
        except Exception: pass
    ids, mask = _encode(prompts)
    if mask is None:
        mask = torch.ones_like(ids)
    n = ids.shape[0]
    gen = [[] for _ in range(n)]
    eos_id = tokenizer.eos_token_id
    done = torch.zeros(n, dtype=torch.bool, device=ids.device)
    row = torch.arange(n, device=ids.device)
    for _ in range(max_new):
        out = _safe_forward(ids, mask)
        logits = _to_logits(out)
        last_idx = _last_real_index(mask)                 # TRUE last real position per row
        nxt = logits[row, last_idx, :].argmax(dim=-1)     # token after "equals " (then after each gen)
        for i in range(n):
            if not done[i]:
                tid = int(nxt[i].item())
                if eos_id is not None and tid == eos_id:
                    done[i] = True
                else:
                    gen[i].append(tid)
        if done.all():
            break
        # Append on the right; with left padding this keeps generated tokens contiguous at
        # the end, and _last_real_index still resolves the correct column on the next step.
        ids = torch.cat([ids, nxt.unsqueeze(1)], dim=1)
        mask = torch.cat([mask, (~done).long().unsqueeze(1)], dim=1)
    return [tokenizer.decode(g, skip_special_tokens=True) for g in gen]

_NUM_RE = re.compile(r"-?\d[\d,]*")
def parse_int(text):
    """Pull the FIRST integer out of a greedy continuation; handle leading spaces, commas
    (1,234), and multi-token splits (already merged by decode()). None if no digits."""
    if text is None:
        return None
    m = _NUM_RE.search(text.strip())
    if not m:
        return None
    s = m.group(0).replace(",", "").rstrip("-")  # guard against stray trailing punctuation
    if s in ("", "-"):
        return None
    try:
        return int(s)
    except ValueError:
        return None

def _parse_to_nan(text):
    v = parse_int(text)
    return float(v) if v is not None else np.nan

# ----------------------------------------------------------------------------------------
# 4) Generic guarded batched evaluator. Caches predictions per artifact key so a disconnect
#    resumes exactly where it stopped (batch-level checkpointing inside the artifact).
#    Prompts for each key are regenerated deterministically (fixed seeds) so a cached
#    prefix stays aligned with the current prompt list across reruns.
# ----------------------------------------------------------------------------------------
def _eval_prompts(prompts, cache_key):
    """Greedy-decode every prompt, return list[str] continuations. Resumable: persists a
    growing list and only decodes the not-yet-done tail."""
    if has_artifact(cache_key):
        cont = load_json(cache_key)
        if len(cont) >= len(prompts):
            log(f"[{cache_key}] cached ({len(cont)} preds) — skipping forward passes.")
            return cont[:len(prompts)]
        # cached prefix shorter than needed -> resume from where it stopped.
    else:
        cont = []
    bs = int(CFG["g3_eval_batch_size"])
    start = len(cont)
    log(f"[{cache_key}] decoding {len(prompts)-start} / {len(prompts)} prompts (resume @ {start})")
    for i in range(start, len(prompts), bs):
        chunk = prompts[i:i + bs]
        cont.extend(_greedy_continuations(chunk))
        save_json(cache_key, cont)   # checkpoint after every batch -> disconnect-safe.
    return cont[:len(prompts)]

# ----------------------------------------------------------------------------------------
# CHECK 1 — ACCURACY on the experimental (0+B)*C stimuli.
# ----------------------------------------------------------------------------------------
if has_artifact("g3_accuracy_result"):
    acc_res = load_json("g3_accuracy_result")
else:
    prompts = [s["prompt"] for s in STIM]
    conts = _eval_prompts(prompts, "g3_pred_experimental")
    preds = [parse_int(c) for c in conts]
    correct = [int(p is not None and p == s["answer"]) for p, s in zip(preds, STIM)]
    overall = float(np.mean(correct)) if correct else 0.0
    parsed_rate = float(np.mean([p is not None for p in preds])) if preds else 0.0
    acc_res = {"overall_accuracy": overall, "parsed_rate": parsed_rate,
               "n": len(STIM), "floor": ACC_FLOOR,
               "examples": [{"prompt": s["prompt"], "pred": p, "gold": s["answer"]}
                            for s, p in list(zip(STIM, preds))[:8]]}
    save_json("g3_accuracy_result", acc_res)
log(f"CHECK1 ACCURACY: overall={acc_res['overall_accuracy']:.3f} "
    f"(parsed_rate={acc_res['parsed_rate']:.3f}, n={acc_res['n']}, floor={ACC_FLOOR})")

# ----------------------------------------------------------------------------------------
# CHECK 2 — NO-OP CHECK (guards the additive-identity correction).
#   (a) Hold structure fixed, sweep B,C on a grid; confirm prediction TRACKS B*C.
#   (b) Confirm "0 +" prefix does NOT make the model ignore the multiplication: compare
#       the (0+B)*C surface against a bare "B times C" surface on the same (B,C) grid;
#       both must track B*C and agree, i.e. the additive identity is a true no-op.
# ----------------------------------------------------------------------------------------
if has_artifact("g3_noop_result"):
    noop_res = load_json("g3_noop_result")
else:
    rng = np.random.default_rng(seed + 7)
    g = int(CFG["g3_noop_grid"])
    lo, hi = 2, 99                      # mid-range operands so signal isn't size-limited.
    Bs = sorted(set(int(x) for x in rng.integers(lo, hi + 1, size=g)))
    Cs = sorted(set(int(x) for x in rng.integers(lo, hi + 1, size=g)))
    pairs = [(B, C) for B in Bs for C in Cs]
    bc = np.array([B * C for (B, C) in pairs], dtype=float)

    # (a) experimental surface "(0+B)*C"
    p_exp = [_fmt_expr(B, C) for (B, C) in pairs]
    c_exp = _eval_prompts(p_exp, "g3_pred_noop_exp")
    y_exp = np.array([_parse_to_nan(t) for t in c_exp])

    # (b) bare-multiplication control "B times C" (no additive identity)
    def _fmt_bare(B, C): return f"{B} times {C} equals "
    p_bare = [_fmt_bare(B, C) for (B, C) in pairs]
    c_bare = _eval_prompts(p_bare, "g3_pred_noop_bare")
    y_bare = np.array([_parse_to_nan(t) for t in c_bare])

    def _track_stats(y):
        ok = np.isfinite(y)
        n_ok = int(ok.sum())
        if n_ok < 3:
            return {"corr_with_BC": None, "exact_match_rate": 0.0, "n_parsed": n_ok}
        corr = float(np.corrcoef(y[ok], bc[ok])[0, 1])
        exact = float(np.mean(y[ok] == bc[ok]))
        return {"corr_with_BC": corr, "exact_match_rate": exact, "n_parsed": n_ok}

    s_exp, s_bare = _track_stats(y_exp), _track_stats(y_bare)
    # additive-identity equivalence: do (0+B)*C and B*C give the same answer per pair?
    both = np.isfinite(y_exp) & np.isfinite(y_bare)
    agree_rate = float(np.mean(y_exp[both] == y_bare[both])) if both.sum() else 0.0
    noop_res = {"n_pairs": len(pairs), "exp": s_exp, "bare": s_bare,
                "additive_identity_agree_rate": agree_rate,
                "Bs": Bs, "Cs": Cs}
    save_json("g3_noop_result", noop_res)
log(f"CHECK2 NO-OP: exp corr(B*C)={noop_res['exp']['corr_with_BC']}, "
    f"bare corr(B*C)={noop_res['bare']['corr_with_BC']}, "
    f"additive-identity agree={noop_res['additive_identity_agree_rate']:.3f}")

# ----------------------------------------------------------------------------------------
# CHECK 3 — MUST-COMPUTE (lookup vs computation): accuracy as a function of operand size.
#   Sample per-bin stimuli (reusing the experimental surface), compute per-bin accuracy,
#   and LOCK the contiguous band that shows graceful DEGRADATION:
#     - every bin in band inside [band_lo, band_hi)   (not chance, not lookup),
#     - accuracy generally non-increasing across the run, AND
#     - a real end-to-end DROP (first - last >= g3_min_band_drop) so a FLAT curve cannot
#       masquerade as 'computation'.
# ----------------------------------------------------------------------------------------
if has_artifact("g3_operand_curve"):
    curve = load_json("g3_operand_curve")
else:
    rng = np.random.default_rng(seed + 13)
    bins = [tuple(b) for b in CFG["g3_operand_bins"]]
    per = int(CFG["g3_per_bin_n"])
    rows = []
    for bi, (blo, bhi) in enumerate(bins):
        # sample fresh controlled stimuli for this bin (cached via per-bin pred artifact)
        Bs = rng.integers(blo, bhi + 1, size=per).tolist()
        Cs = rng.integers(blo, bhi + 1, size=per).tolist()
        prompts = [_fmt_expr(int(B), int(C)) for B, C in zip(Bs, Cs)]
        golds = [int(B) * int(C) for B, C in zip(Bs, Cs)]
        conts = _eval_prompts(prompts, f"g3_pred_bin_{bi}")
        preds = [parse_int(c) for c in conts]
        corr = [int(p is not None and p == g) for p, g in zip(preds, golds)]
        acc = float(np.mean(corr)) if corr else 0.0
        rows.append({"bin": bi, "lo": blo, "hi": bhi,
                     "max_operand": bhi, "accuracy": acc, "n": per,
                     "parsed_rate": float(np.mean([p is not None for p in preds]))})
        log(f"  bin {bi} operands[{blo},{bhi}] acc={acc:.3f}")
    curve = {"bins": rows}
    save_json("g3_operand_curve", curve)

# --- LOCK the band: longest contiguous run of bins with band_lo <= acc < band_hi, then
#     require a genuine downward trend AND a real drop across that run. ---
b_lo, b_hi = float(CFG["g3_band_lo"]), float(CFG["g3_band_hi"])
accs = [r["accuracy"] for r in curve["bins"]]
in_band = [(b_lo <= a < b_hi) for a in accs]
# find longest contiguous True run
best = (0, -1, -1)  # (length, start, end)
i = 0
while i < len(in_band):
    if in_band[i]:
        j = i
        while j + 1 < len(in_band) and in_band[j + 1]:
            j += 1
        if (j - i + 1) > best[0]:
            best = (j - i + 1, i, j)
        i = j + 1
    else:
        i += 1
_, lstart, lend = best
locked = None
all_lookup = all(a >= b_hi for a in accs) if accs else False   # flat-100% memorization signal
all_chance = all(a < b_lo for a in accs) if accs else False
if best[0] >= 1:
    lo_operand = curve["bins"][lstart]["lo"]
    hi_operand = curve["bins"][lend]["hi"]
    band_accs = accs[lstart:lend + 1]
    # non-increasing (allow tiny noise) ...
    non_increasing = all(band_accs[k] >= band_accs[k + 1] - 0.05
                         for k in range(len(band_accs) - 1))
    # ... AND a real end-to-end drop so a FLAT band is NOT accepted as computation.
    end_to_end_drop = (band_accs[0] - band_accs[-1]) if len(band_accs) >= 2 else 0.0
    degrading = bool(non_increasing and end_to_end_drop >= MIN_DROP)
    locked = {"operand_lo": int(lo_operand), "operand_hi": int(hi_operand),
              "bin_start": int(lstart), "bin_end": int(lend),
              "bin_accuracies": [float(a) for a in band_accs],
              "non_increasing": bool(non_increasing),
              "end_to_end_drop": float(end_to_end_drop),
              "min_required_drop": MIN_DROP,
              "graceful_degradation": degrading,
              "band_lo": b_lo, "band_hi": b_hi,
              "accuracy_floor": ACC_FLOOR}

# ----------------------------------------------------------------------------------------
# 5) PLOT accuracy vs operand size (inline) + save the figure.
# ----------------------------------------------------------------------------------------
xs = [r["max_operand"] for r in curve["bins"]]
ys = [r["accuracy"] for r in curve["bins"]]
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(xs, ys, "o-", color="#1f77b4", label="accuracy")
ax.axhline(b_lo, ls=":", c="grey", lw=1); ax.axhline(b_hi, ls=":", c="grey", lw=1)
ax.axhline(ACC_FLOOR, ls="--", c="green", lw=1, label=f"floor={ACC_FLOOR}")
if locked is not None:
    ax.axvspan(curve["bins"][locked["bin_start"]]["lo"],
               curve["bins"][locked["bin_end"]]["hi"],
               color="orange", alpha=0.15, label="locked band")
ax.set_xscale("log"); ax.set_xlabel("max operand magnitude (log)")
ax.set_ylabel("greedy-decode accuracy"); ax.set_ylim(-0.02, 1.02)
ax.set_title("Phase 3 / G3 — accuracy vs operand size (must-compute)")
ax.legend(loc="best", fontsize=8); fig.tight_layout()
try:
    fig.savefig(str(ART / "g3_operand_curve.png"), dpi=120)
except Exception as e:
    log(f"(plot save skipped: {e})")
plt.show()

# ----------------------------------------------------------------------------------------
# 6) GATE G3 verdict. PASS requires all three checks:
#    (1) overall accuracy >= floor (compute happens at all),
#    (2) no-op: predictions track B*C AND additive identity is a genuine no-op,
#    (3) must-compute: a graceful-DEGRADATION band exists (real drop) -> LOCK + save spec.
# ----------------------------------------------------------------------------------------
c1 = acc_res["overall_accuracy"] >= ACC_FLOOR

exp_corr = noop_res["exp"]["corr_with_BC"]
c2_track = (exp_corr is not None and exp_corr >= 0.90 and
            noop_res["exp"]["exact_match_rate"] >= 0.50)
c2_noop = noop_res["additive_identity_agree_rate"] >= 0.90    # "0 +" is a true no-op.
c2 = bool(c2_track and c2_noop)

c3 = bool(locked is not None and locked.get("graceful_degradation", False))

g3_pass = bool(c1 and c2 and c3)

if g3_pass and locked is not None:
    # Save the SINGLE locked-band spec that Phases 4-9 consume.
    band_spec = {"operand_lo": locked["operand_lo"], "operand_hi": locked["operand_hi"],
                 "accuracy_floor": ACC_FLOOR, "band_lo": b_lo, "band_hi": b_hi,
                 "bin_accuracies": locked["bin_accuracies"],
                 "end_to_end_drop": locked["end_to_end_drop"],
                 "source_stimuli": src_name, "seed": seed,
                 "overall_accuracy": acc_res["overall_accuracy"]}
    save_json("locked_band_spec", band_spec)
    CFG["locked_operand_lo"] = locked["operand_lo"]
    CFG["locked_operand_hi"] = locked["operand_hi"]
    log(f"LOCKED BAND saved: operands [{locked['operand_lo']}, {locked['operand_hi']}] "
        f"(drop={locked['end_to_end_drop']:.2f}) -> artifact 'locked_band_spec' (Phases 4-9).")

detail = (f"acc={acc_res['overall_accuracy']:.3f}/floor={ACC_FLOOR} (C1={'P' if c1 else 'F'}); "
          f"no-op exp_corr={exp_corr}, agree={noop_res['additive_identity_agree_rate']:.2f} "
          f"(C2={'P' if c2 else 'F'}); "
          f"locked={None if locked is None else (locked['operand_lo'], locked['operand_hi'])}, "
          f"drop={None if locked is None else round(locked['end_to_end_drop'],3)}, "
          f"graceful={None if locked is None else locked['graceful_degradation']} "
          f"(C3={'P' if c3 else 'F'})")
set_gate("G3", g3_pass, detail)

print("\n================= PHASE 3 / GATE G3 =================")
print(f"CHECK 1 ACCURACY     : {'PASS' if c1 else 'FAIL'}  "
      f"(overall={acc_res['overall_accuracy']:.3f} >= floor {ACC_FLOOR}?)")
print(f"CHECK 2 NO-OP        : {'PASS' if c2 else 'FAIL'}  "
      f"(track B*C corr={exp_corr}, exact={noop_res['exp']['exact_match_rate']:.2f}; "
      f"additive-identity no-op agree={noop_res['additive_identity_agree_rate']:.2f})")
print(f"CHECK 3 MUST-COMPUTE : {'PASS' if c3 else 'FAIL'}  "
      f"(graceful-degradation band={'none' if locked is None else (locked['operand_lo'], locked['operand_hi'])}"
      f"{'' if locked is None else f", drop={locked['end_to_end_drop']:.2f}>={MIN_DROP}"})")
print("-----------------------------------------------------")
print("per-bin accuracy vs operand size:")
for r in curve["bins"]:
    mark = ""
    if locked is not None and locked["bin_start"] <= r["bin"] <= locked["bin_end"]:
        mark = "  <-- LOCKED"
    print(f"   operands[{r['lo']:>3},{r['hi']:>3}]  acc={r['accuracy']:.3f}  n={r['n']}{mark}")
print("-----------------------------------------------------")
print(f"GATE G3: {'PASS' if g3_pass else 'FAIL'}")
if not g3_pass:
    print("\nFAIL GUIDANCE:")
    if not c1:
        print(" - Accuracy below floor. The base model may not do multi-digit arithmetic")
        print("   greedily. Try: (a) the -Instruct model, (b) few-shot prompting, then RE-RUN")
        print("   the Phase 2 token-/answer-control gate (G2) on the new surface form.")
    if not c2:
        if not c2_track:
            print(" - Predictions don't track B*C. Stimulus surface may not elicit the product")
            print("   (tokenization of 'times', wrong answer cue). Revisit Phase 2 surface.")
        if not c2_noop:
            print(" - '0 +' is NOT a no-op (additive-identity surface disagrees with bare B*C).")
            print("   The additive-identity correction is unjustified on this model — reconsider.")
    if not c3:
        if all_lookup:
            print(" - Accuracy pinned at/above band_hi in EVERY bin -> looks like memorized")
            print("   lookup. GROW the range (add larger operand bins) until it degrades.")
        elif all_chance:
            print(" - Accuracy below band_lo everywhere -> collapsed to chance (no computation).")
            print("   SHRINK the range (lower the upper operand bins) until signal appears.")
        elif locked is not None and not locked["graceful_degradation"]:
            print(f" - A band sits in [{b_lo},{b_hi}) but is FLAT (end-to-end drop "
                  f"{locked['end_to_end_drop']:.2f} < required {MIN_DROP}); flat != computation.")
            print("   Widen the operand-size span so real degradation shows across bins.")
        else:
            print(" - No contiguous graceful-degradation band. Adjust g3_operand_bins span.")
print("=====================================================")

---
# Phase 4 — Token‑boundary mapping (enables the patches)

Not a gate, but getting it wrong silently corrupts every patch. Because the stimuli are **token‑aligned by construction** (guaranteed by the Phase 2 assertions), the index map is **shared across a whole template**, not computed per example.

`token_map(template, condition, pad_len)` returns the canonical token indices: where the intermediate value (`B*C`, or the held `B` after `0+B`) first becomes decodable, the critical `*` operator, the index where the structural role flips between the depth conditions, and the deterministic operand‑index shift as a function of pad length. Unit‑tested against hand‑verified examples; later phases import it and never recompute boundaries ad hoc.


In [ ]:
# ============================================================================
# Phase 4 — Token-boundary mapping (NOT a gate, but it silently corrupts every
#           patch if wrong). Pure tokenizer / CPU.
# ----------------------------------------------------------------------------
# Reconciled to Phase 2's PARENTHESIZED additive-identity templates + SUFFIX
# padding (must match Phase 2's surface EXACTLY or every patch is mis-indexed):
#   depth_left  : "( 0 + B ) * C ="   -> (0+B)*C   ; '*' at paren-depth 0
#   depth_right : "0 + ( B * C ) ="   -> 0+(B*C)   ; '*' at paren-depth 1
#   suffix pad k: append " + 0" * k before "=" (grows length, not '*'-depth).
#
# Two layers of mapping:
#   (1) token_map(template, condition, pad_len) -> CANONICAL indices for the
#       hand-verifiable SINGLE-TOKEN-operand probe (B='3', C='5'). Shared across
#       all single-token-operand examples of a template (the spec's "patch fixed
#       indices" payoff). eq index shifts by +2k under suffix padding; the core
#       operand/operator indices are INVARIANT to k (padding is suffixed).
#   (2) token_map_for_record(rec) -> the EXACT per-example indices Phase 2 already
#       stored, the robust path when operands are multi-token (their token length,
#       hence '*'/C positions, varies example to example).
#
# ADVERSARIAL-REVIEW-style safeguards kept from the prior version:
#   * BOS offset prefers Phase 2's persisted value (CFG['bos_offset'] / artifact);
#     detection is only a fallback and disagreement is logged loudly.
#   * Unit tests locate operator/operands by TOKEN CONTENT in an INDEPENDENT
#     re-tokenization (not by reusing the implementation's offsets), so an
#     off-by-one in the core layout OR the +2k suffix shift actually fails.
# ============================================================================

import json

# Single-token probe operands (single digits are 1 token in Llama BPE) so the
# canonical map is well-defined and shared. Distinct from each other and from the
# pad filler '0' so content-location is unambiguous even under padding.
_PROBE = {"B": "3", "C": "5"}
_SEP = " "
_CUE = "="
_PAD_UNIT = ["+", "0"]   # one suffix identity op; MUST match Phase 2's PAD_UNIT.

# ---- cross-check Phase 2's surface convention so the two cells cannot drift ----
if has_artifact("phase2_surface_spec", "json"):
    _spec = load_json("phase2_surface_spec")
    if _spec.get("separator") != _SEP or _spec.get("answer_cue") != _CUE \
            or _spec.get("pad_style") != "suffix_before_eq":
        log(f"Phase 4 WARNING: phase2_surface_spec {_spec} disagrees with this cell's "
            f"render convention (sep={_SEP!r}, cue={_CUE!r}, suffix pad). Indices may be wrong.")
    else:
        log("Phase 4: surface convention matches Phase 2 (sep/cue/suffix-pad OK).")
else:
    log("Phase 4: no phase2_surface_spec on disk; using built-in render convention.")

# ---------------------------------------------------------------- render (== Phase 2) ----
def _render(template, pad_len=0, B=None, C=None):
    B = _PROBE["B"] if B is None else str(int(B))
    C = _PROBE["C"] if C is None else str(int(C))
    if template == "depth_left":
        toks = ["(", "0", "+", B, ")", "*", C]
    elif template == "depth_right":
        toks = ["0", "+", "(", B, "*", C, ")"]
    else:
        raise ValueError(f"unknown template {template!r}")
    for _ in range(int(pad_len)):
        toks += _PAD_UNIT[:]            # " + 0" suffix identity op
    toks += [_CUE]
    return _SEP.join(toks)

# ---------------------------------------------------------------- BOS handling ----
def _detect_bos_offset():
    a = tokenizer("0", add_special_tokens=True)["input_ids"]
    b = tokenizer("0", add_special_tokens=False)["input_ids"]
    return max(0, len(a) - len(b))

def _bos_offset():
    declared = None
    if isinstance(CFG, dict) and "bos_offset" in CFG:
        declared = int(CFG["bos_offset"])
    elif has_artifact("phase2_bos_offset", "json"):
        declared = int(load_json("phase2_bos_offset"))
    detected = _detect_bos_offset()
    if declared is not None:
        if declared != detected:
            log(f"Phase 4 WARNING: detected BOS offset {detected} != Phase 2 declared "
                f"{declared}; using Phase 2's {declared} (its tokenization is source of truth).")
        return declared
    return detected

# ---------------------------------------------------------------- content locator ----
def _toks_with_specials(text):
    ids = tokenizer(text, add_special_tokens=True)["input_ids"]
    return ids, [tokenizer.decode([i]).strip() for i in ids]

def _first(toks, sym, start=0):
    for i in range(start, len(toks)):
        if toks[i] == sym:
            return i
    return None

def _resolve_canonical(template):
    """Locate canonical single-token-operand indices BY CONTENT in the BOS-prefixed
    pad_len=0 surface. Returns absolute indices (BOS already included)."""
    text = _render(template, pad_len=0)
    ids, toks = _toks_with_specials(text)
    op0 = _first(toks, "0")                      # first '0' is the additive identity op0
    B   = _first(toks, _PROBE["B"])
    C   = _first(toks, _PROBE["C"])
    star = _first(toks, "*")
    eq  = _first(toks, "=")
    for nm, v in (("op0", op0), ("B", B), ("C", C), ("star", star), ("eq", eq)):
        assert v is not None, f"{template}: could not locate {nm} in {toks!r}"
    return {"op0": op0, "B": B, "C": C, "star": star, "eq": eq,
            "core_len": len(ids), "toks": toks}

# ---------------------------------------------------------------- build & cache ----
if has_artifact("phase4_token_map", "json"):
    _MAP = load_json("phase4_token_map")
    log("Phase 4: loaded cached token-boundary map.")
else:
    _bos = _bos_offset()
    _MAP = {
        "bos_offset": _bos,
        "pad_style": "suffix_before_eq",     # eq shifts +len(PAD_UNIT)*k ; core invariant
        "pad_unit_len": len(_PAD_UNIT),
        "templates": {t: {k: v for k, v in _resolve_canonical(t).items() if k != "toks"}
                      for t in ("depth_left", "depth_right")},
        "notes": "Absolute indices into the BOS-prefixed sequence for SINGLE-TOKEN operands "
                 "(B='3',C='5'). Suffix pad k appends k*'+ 0' before '='; core indices are "
                 "pad-invariant, eq index += pad_unit_len*k. Multi-token operands: use "
                 "token_map_for_record(rec).",
    }
    save_json("phase4_token_map", _MAP)
    log(f"Phase 4: token-boundary map saved (bos_offset={_bos}).")

# ---------------------------------------------------------------- exported API ----
def token_map(template, condition=None, pad_len=0):
    """Canonical single-token-operand indices for a locked template, into the
    BOS-prefixed, suffix-padded sequence:
      - probed_operand          : B index (held FIXED across the depth pair).
      - critical_operator       : '*' index (its binding differs between conditions).
      - intermediate_decodable  : C index (last operand; B*C / held value decodable here).
      - role_flip               : op0 index (the additive-identity '0').
      - answer_cue              : '=' index (shifts by pad_unit_len*pad_len).
    `condition` is accepted for call-site symmetry; the canonical map is condition-free."""
    if template not in _MAP["templates"]:
        raise ValueError(f"unknown template {template!r}; expected {list(_MAP['templates'])}")
    L = _MAP["templates"][template]
    k = int(pad_len)
    shift = _MAP["pad_unit_len"] * k
    return {
        "probed_operand":        L["B"],
        "critical_operator":     L["star"],
        "intermediate_decodable": L["C"],
        "role_flip":             L["op0"],
        "answer_cue":            L["eq"] + shift,
        "core_len":              L["core_len"],
        "bos_offset":            _MAP["bos_offset"],
        "pad_len":               k,
    }

def token_map_for_record(rec):
    """Exact per-example indices straight from the Phase 2 record (robust for
    MULTI-token operands, whose '*'/C positions vary by operand token length)."""
    oi = rec.get("operand_token_indices", {})
    return {
        "probed_operand":         oi.get("B"),
        "critical_operator":      rec.get("operator_token_index"),
        "intermediate_decodable": oi.get("C"),
        "role_flip":              rec.get("op0_token_index"),
        "answer_cue":             rec.get("eq_token_index"),
        "pad_len":                rec.get("pad_len", 0),
    }

# ---------------------------------------------------------------- UNIT TESTS ----
def _reference_indices(template, pad_len):
    """INDEPENDENT ground truth: re-tokenize the padded surface and locate by CONTENT,
    WITHOUT using token_map's stored offsets -> a real off-by-one fails here."""
    ids, toks = _toks_with_specials(_render(template, pad_len=pad_len))
    star = _first(toks, "*")
    op0  = _first(toks, "0")
    B    = _first(toks, _PROBE["B"])
    C    = _first(toks, _PROBE["C"])
    eq   = len(toks) - 1                          # '=' is the final token of the surface
    assert toks[eq] == "=", f"{template} k={pad_len}: last token not '=': {toks!r}"
    return {"probed_operand": B, "critical_operator": star,
            "intermediate_decodable": C, "role_flip": op0, "answer_cue": eq}

def _test_token_map():
    fails = []
    def ck(name, got, want):
        ok = (got == want)
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}: got {got}, want {want}")
        if not ok:
            fails.append(name)

    pad_lens = sorted(set([0] + [int(k) for k in CFG.get("pad_lengths", [2, 4, 8])]))

    # (1) NON-TAUTOLOGICAL: token_map must equal the independent content-anchored
    #     reference for every template and every pad length.
    for tmpl in ("depth_left", "depth_right"):
        for k in pad_lens:
            ref = _reference_indices(tmpl, k)
            got = token_map(tmpl, tmpl.split("_")[1], k)
            for key in ("probed_operand", "critical_operator",
                        "intermediate_decodable", "role_flip", "answer_cue"):
                ck(f"{tmpl}.{key}(k={k})", got[key], ref[key])

    # (2) The spec's core invariants of the parenthesized contrast:
    #     B (probed_operand) index EQUAL across conditions; '*'-depth differs so the
    #     critical_operator index differs; answer_cue equal at k=0.
    l0, r0 = token_map("depth_left", "left", 0), token_map("depth_right", "right", 0)
    ck("B index equal across conditions (k=0)", l0["probed_operand"], r0["probed_operand"])
    ck("answer_cue equal across conditions (k=0)", l0["answer_cue"], r0["answer_cue"])
    print(f"  [INFO] critical_operator differs across conditions: "
          f"left={l0['critical_operator']} vs right={r0['critical_operator']} "
          f"({'OK' if l0['critical_operator'] != r0['critical_operator'] else 'UNEXPECTED-SAME'})")

    # (3) Pure suffix-shift property: only answer_cue moves, by pad_unit_len*k.
    base = token_map("depth_right", "right", 0)
    for k in [x for x in pad_lens if x > 0]:
        m = token_map("depth_right", "right", k)
        ck(f"core invariant under pad (B,k={k})", m["probed_operand"], base["probed_operand"])
        ck(f"core invariant under pad (*,k={k})", m["critical_operator"], base["critical_operator"])
        ck(f"answer_cue shift == pad_unit_len*k (k={k})",
           m["answer_cue"] - base["answer_cue"], _MAP["pad_unit_len"] * k)

    # (4) Absolute hand-verified anchor for bos_offset==1, single-token operands:
    #     depth_right: BOS 0 + ( 3 * 5 ) =  -> op0=1,B=4,*=5,C=6,eq=8
    if _MAP["bos_offset"] == 1:
        m = token_map("depth_right", "right", 0)
        ck("abs.role_flip", m["role_flip"], 1)
        ck("abs.probed_operand", m["probed_operand"], 4)
        ck("abs.critical_operator", m["critical_operator"], 5)
        ck("abs.intermediate_decodable", m["intermediate_decodable"], 6)
        ck("abs.answer_cue", m["answer_cue"], 8)
    else:
        print(f"  [INFO] bos_offset={_MAP['bos_offset']} != 1; absolute anchor skipped "
              f"(content-anchored ref above still enforces correctness).")

    # (5) token_map_for_record agrees with Phase 2 records (if dataset present).
    if has_artifact("phase2_stimuli", "json"):
        recs = load_json("phase2_stimuli")
        depth_lefts = [r for r in recs if r.get("condition") == "depth_left"][:1]
        if depth_lefts:
            r = depth_lefts[0]
            tm = token_map_for_record(r)
            ck("record.probed_operand matches stored B index",
               tm["probed_operand"], r["operand_token_indices"]["B"])
            ck("record.critical_operator matches stored * index",
               tm["critical_operator"], r["operator_token_index"])

    # (6) unknown template raises
    raised = False
    try:
        token_map("depth_middle", "x", 0)
    except ValueError:
        raised = True
    ck("unknown_template_raises", raised, True)

    print(f"Phase 4 token_map unit tests: {'ALL PASS' if not fails else 'FAIL -> ' + ', '.join(fails)}")
    assert not fails, f"token_map unit tests failed: {fails}"

_test_token_map()
log("Phase 4: token-boundary map ready; later phases import token_map / "
    "token_map_for_record (never recompute boundaries ad hoc).")


---
# Phase 5 — Pipeline validation on a known result · **Gate G4**

Prove the instrument works where the answer is **known** before trusting it on the novel question. Reproduce one established arithmetic‑circuit finding (single‑addition activation patching: information concentrates at the **last token** in **mid‑to‑late layers**) through *this* patching pipeline.

Validates that (a) hooks read the right activations, (b) the patch direction (clean → corrupted) is implemented correctly, and (c) the effect metric (logit difference) has the right magnitude **and sign**. The layer‑resolved effect is asserted to land in the expected region. **A green G4 is the license to trust Phase 6 onward** — do not proceed to novel probes until it passes.


In [ ]:
# Phase 5 — Gate G4: pipeline validation on a KNOWN arithmetic-circuit result
# (activation patching of "A + B =" addition) BEFORE trusting the instrument on the novel question.
#
# PRIOR RESULT TARGETED (ground truth we check against):
#   Stolfo, Belinkov & Sachan (2023, EMNLP), "A Mechanistic Interpretation of Arithmetic
#   Reasoning in LMs using Causal Mediation Analysis", + the bag-of-heuristics line
#   (Nikankin et al. 2024). Established localization: query-relevant info is moved from
#   mid-sequence EARLY layers to the LAST token via attention, then LATE-layer MLPs write
#   the result into the residual stream at the LAST token. We therefore expect the
#   activation-patching effect (clean->corrupted residual patch at the final token) to be
#   LARGE and POSITIVE in roughly the back half of the network at the LAST token.
#   (Hook name blocks.{L}.hook_resid_post and the run_with_hooks(corrupted, fwd_hooks=[...])
#    patching idiom verified against TransformerLens docs.)
#
# PATCH DIRECTION (stated explicitly, do not flip) — UNCHANGED, verified correct:
#   We CACHE CLEAN activations, run the CORRUPTED prompt, and WRITE the clean residual
#   stream in at one (layer, position). i.e. restore the clean computation into the
#   corrupted run = denoising / "noising-recovery" patching: a position that carries the
#   answer info, when restored, pushes the logit-diff back toward the clean value.
#
# METRIC SIGN (stated explicitly) — UNCHANGED, verified consistent with the direction:
#   logit_diff = logit(clean_answer_token) - logit(corrupted_answer_token) at FINAL pos.
#   clean run -> large POSITIVE ; corrupted run -> low/NEGATIVE ; a good patch RAISES it.
#   recovery in [0,1]: 0 = no better than corrupted, 1 = fully restored to clean.
#
# CORRECTNESS FIX (load-bearing): the prompts must NOT end in a trailing space. On Llama-3's
#   tiktoken tokenizer a trailing space at end-of-string becomes its own token, so the model
#   would then predict the BARE digit "7" (a different vocab id than " 7"), making the metric
#   read the wrong ids and the argmax sanity-check fail spuriously. We end the prompt at "is"
#   so the true next token is " 7"/" 8" (matching _single_token_id's leading-space convention),
#   and we additionally re-derive the answer ids empirically from the model's top predictions.
#
# RESILIENCE: the (layer x position) sweep is checkpointed PER LAYER to disk, so a GPU
#   disconnect mid-sweep never discards completed layers. Re-running the cell reloads finished
#   layers and only computes the missing ones.

import numpy as np
import torch

# ---- set_gate: documented helpers have no gate registry, so define one defensively,
# ---- persisting via the documented save_json/load_json.
def set_gate(name, passed, detail=""):
    gates = load_json('gates') if has_artifact('gates') else {}
    gates[name] = {"pass": bool(passed), "detail": str(detail)}
    save_json('gates', gates)
    log(f"GATE {name}: {'PASS' if passed else 'FAIL'} — {detail}")
    return gates[name]

# --------------------------------------------------------------------------------
# 1) Clean / corrupted prompt pair with KNOWN, single-token, DIFFERING answers.
#    NO trailing space: the final token is "is", and the model predicts " 7"/" 8".
#    "The sum of 3 and 4 is" -> " 7"  (clean)
#    "The sum of 3 and 5 is" -> " 8"  (corrupted; only the second operand changes)
#    Token-length matched so positions line up 1:1 for patching.
CLEAN_PROMPT     = "The sum of 3 and 4 is"
CORRUPTED_PROMPT = "The sum of 3 and 5 is"
CLEAN_ANSWER     = "7"   # answer for the CLEAN prompt
CORRUPTED_ANSWER = "8"   # answer for the CORRUPTED prompt

def _single_token_id(s):
    # Answer tokens in arithmetic prompts are emitted with a leading space.
    ids = tokenizer(" " + s, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, f"answer {s!r} is not a single token: {ids}"
    return ids[0]

clean_ans_id     = _single_token_id(CLEAN_ANSWER)
corrupted_ans_id = _single_token_id(CORRUPTED_ANSWER)

# Tokenize prompts (TL prepends BOS by default; both handled identically).
clean_tokens     = model.to_tokens(CLEAN_PROMPT)       # [1, seq]
corrupted_tokens = model.to_tokens(CORRUPTED_PROMPT)   # [1, seq]
assert clean_tokens.shape == corrupted_tokens.shape, \
    f"prompt length mismatch: {clean_tokens.shape} vs {corrupted_tokens.shape} — positions must align for patching"
seq_len   = clean_tokens.shape[1]
final_pos = seq_len - 1
n_layers  = model.cfg.n_layers

# Guard: the final prompt token must NOT itself be a lone space (would break the
# answer-token convention). 'is' should be the last token.
_last_tok_str = tokenizer.decode([clean_tokens[0, final_pos].item()])
assert _last_tok_str.strip() != "", (
    f"final prompt token decodes to whitespace {_last_tok_str!r}; remove the trailing "
    f"space from the prompt so the model predicts a leading-space answer token")
log(f"final prompt token = {_last_tok_str!r} (expected non-space, e.g. 'is')")

# Sanity: confirm exactly one token differs (the corrupted operand) — else position
# alignment / single-operand-corruption assumption is violated.
_diff = (clean_tokens[0] != corrupted_tokens[0]).nonzero().flatten().tolist()
log(f"differing token positions (clean vs corrupted): {_diff} (expect exactly one operand token)")
assert len(_diff) == 1, f"expected a single corrupted operand token, got positions {_diff}"

# --------------------------------------------------------------------------------
# 2) Metric: logit_diff at the FINAL position = logit(clean_ans) - logit(corrupted_ans).
def logit_diff_from_logits(logits, c_id, k_id):
    final = logits[0, final_pos]               # FINAL-position next-token logits
    return (final[c_id] - final[k_id]).item()

# Baselines: clean & corrupted forward passes. We also EMPIRICALLY re-derive the answer
# ids from the model's actual top predictions and reconcile with the leading-space ids,
# so a tokenizer surprise can't silently produce a meaningless metric.
if has_artifact('g4_baselines'):
    g4_base = load_json('g4_baselines')
    clean_ans_id      = g4_base['clean_ans_id']
    corrupted_ans_id  = g4_base['corrupted_ans_id']
    clean_baseline    = g4_base['clean']
    corrupted_baseline = g4_base['corrupted']
    log("G4 baselines loaded from cache.")
else:
    model.eval()
    with torch.no_grad():
        # Cache ONLY resid_post hooks (memory-safe on an 8B model).
        clean_logits, clean_cache = model.run_with_cache(
            clean_tokens, names_filter=lambda n: n.endswith("hook_resid_post"))
        corrupted_logits = model(corrupted_tokens)

    clean_top     = clean_logits[0, final_pos].argmax().item()
    corrupted_top = corrupted_logits[0, final_pos].argmax().item()
    log(f"empirical top tokens — clean: {tokenizer.decode([clean_top])!r}  "
        f"corrupted: {tokenizer.decode([corrupted_top])!r}")

    # The model must actually KNOW this fact, and the two answers must differ; otherwise
    # the 'known result' premise is broken and the gate is meaningless.
    assert tokenizer.decode([clean_top]).strip() == CLEAN_ANSWER, (
        f"model's top clean prediction is {tokenizer.decode([clean_top])!r}, not "
        f"{CLEAN_ANSWER!r}; pick a fact the model gets right before patching")
    assert tokenizer.decode([corrupted_top]).strip() == CORRUPTED_ANSWER, (
        f"model's top corrupted prediction is {tokenizer.decode([corrupted_top])!r}, not "
        f"{CORRUPTED_ANSWER!r}; corruption did not flip the answer as expected")
    assert clean_top != corrupted_top, "clean and corrupted answers must differ"

    # Reconcile: the leading-space ids from _single_token_id MUST match the model's
    # actual emitted answer tokens; if not, trust the empirical ids (and warn).
    if clean_top != clean_ans_id or corrupted_top != corrupted_ans_id:
        log(f"WARNING: leading-space ids ({clean_ans_id},{corrupted_ans_id}) != empirical "
            f"top ids ({clean_top},{corrupted_top}); using empirical ids for the metric.")
        clean_ans_id, corrupted_ans_id = clean_top, corrupted_top

    clean_baseline     = logit_diff_from_logits(clean_logits, clean_ans_id, corrupted_ans_id)
    corrupted_baseline = logit_diff_from_logits(corrupted_logits, clean_ans_id, corrupted_ans_id)

    # Persist the clean resid stack so a reconnected kernel never needs the model to
    # rebuild the cache for the sweep. Stack -> [n_layers, 1, seq, d_model].
    clean_resid_stack = torch.stack(
        [clean_cache[f"blocks.{L}.hook_resid_post"][0] for L in range(n_layers)], dim=0
    ).to(torch.float16).cpu()
    save_pickle('g4_clean_resid_stack', clean_resid_stack)
    save_json('g4_baselines', {"clean": clean_baseline, "corrupted": corrupted_baseline,
                               "clean_ans_id": int(clean_ans_id),
                               "corrupted_ans_id": int(corrupted_ans_id)})

log(f"clean_baseline (logit_diff) = {clean_baseline:+.3f}  | corrupted_baseline = {corrupted_baseline:+.3f}")
assert clean_baseline > corrupted_baseline, \
    "clean logit_diff must exceed corrupted — metric sign or answer tokens are wrong"

# Normalized recovery: 0 at corrupted_baseline, 1 at clean_baseline.
_denom = (clean_baseline - corrupted_baseline) + 1e-8
def recovery(patched_ld):
    return (patched_ld - corrupted_baseline) / _denom

# --------------------------------------------------------------------------------
# 3) (layer x position) patch sweep — GPU-expensive, CHECKPOINTED PER LAYER to disk.
#    Hook: blocks.{L}.hook_resid_post (EXACT TransformerLens name).
#    Per (L,pos): run CORRUPTED; OVERWRITE resid_post[:,pos,:] at layer L with the CLEAN
#    cached value at the same (L,pos); record final-position logit_diff. clean -> corrupted.
if has_artifact('g4_patch_sweep'):
    patch_recovery = np.array(load_json('g4_patch_sweep'))  # [n_layers, seq_len], normalized
    log(f"G4 patch sweep loaded from cache: shape {patch_recovery.shape}")
else:
    # Resume partial sweep if present, else start fresh.
    if has_artifact('g4_patch_sweep_partial'):
        part = load_json('g4_patch_sweep_partial')
        patch_ld   = np.array(part['patch_ld'], dtype=np.float64)
        layer_done = list(part['layer_done'])
        log(f"resuming partial sweep: {sum(layer_done)}/{n_layers} layers already done")
    else:
        patch_ld   = np.zeros((n_layers, seq_len), dtype=np.float64)
        layer_done = [False] * n_layers

    # Load the persisted clean resid stack (model not required); rebuild only if absent.
    if has_artifact('g4_clean_resid_stack'):
        clean_resid_stack = load_pickle('g4_clean_resid_stack').to(model.cfg.device)
    else:
        with torch.no_grad():
            _, _cc = model.run_with_cache(
                clean_tokens, names_filter=lambda n: n.endswith("hook_resid_post"))
        clean_resid_stack = torch.stack(
            [_cc[f"blocks.{L}.hook_resid_post"][0] for L in range(n_layers)], dim=0
        ).to(model.cfg.device)
        save_pickle('g4_clean_resid_stack', clean_resid_stack.to(torch.float16).cpu())

    def make_patch_hook(clean_resid_layer, pos):
        # clean_resid_layer: CLEAN cached resid_post for this layer, shape [seq, d_model].
        def hook(resid_post, hook):
            # WRITE clean activation into the corrupted run at a single position.
            resid_post[:, pos, :] = clean_resid_layer[pos, :].to(resid_post.dtype)
            return resid_post
        return hook

    model.eval()
    with torch.no_grad():
        for L in range(n_layers):
            if layer_done[L]:
                continue
            hook_name   = f"blocks.{L}.hook_resid_post"   # EXACT TL hook name
            clean_layer = clean_resid_stack[L].to(model.cfg.device)  # [seq, d_model]
            for pos in range(seq_len):
                patched_logits = model.run_with_hooks(
                    corrupted_tokens,
                    fwd_hooks=[(hook_name, make_patch_hook(clean_layer, pos))],
                )
                patch_ld[L, pos] = logit_diff_from_logits(
                    patched_logits, clean_ans_id, corrupted_ans_id)
            layer_done[L] = True
            # CHECKPOINT after every layer so a disconnect loses at most one layer.
            save_json('g4_patch_sweep_partial',
                      {"patch_ld": patch_ld.tolist(), "layer_done": layer_done})
            log(f"swept layer {L+1}/{n_layers} (checkpointed)")

    assert all(layer_done), "sweep incomplete but exited loop — investigate"
    patch_recovery = (patch_ld - corrupted_baseline) / _denom  # normalized [~0, ~1]
    save_json('g4_patch_sweep', patch_recovery.tolist())
    log("G4 patch sweep computed and cached.")

# --------------------------------------------------------------------------------
# 4) Heatmap of the effect (normalized recovery): rows = layers, cols = positions.
try:
    import matplotlib.pyplot as plt
    pos_labels = [tokenizer.decode([t]).replace("\n", "\\n") for t in clean_tokens[0].tolist()]
    fig, ax = plt.subplots(figsize=(max(8, seq_len * 0.7), max(5, n_layers * 0.22)))
    im = ax.imshow(patch_recovery, aspect="auto", origin="lower", cmap="RdBu_r",
                   vmin=-1.0, vmax=1.0)
    ax.set_xlabel("token position (clean -> corrupted resid_post patch)")
    ax.set_ylabel("layer")
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(pos_labels, rotation=60, ha="right", fontsize=8)
    ax.set_title("G4: addition activation patching — normalized logit-diff recovery\n"
                 "(expect bright band at FINAL token, middle-to-late layers — Stolfo 2023)")
    fig.colorbar(im, ax=ax, label="recovery (0=corrupted, 1=clean)")
    ax.axvline(final_pos, color="black", lw=1, ls="--")  # mark the final token column
    plt.tight_layout()
    try:
        fig.savefig(str(ART / "g4_patch_heatmap.png"), dpi=130)  # persist the figure too
    except Exception as _e:
        log(f"(heatmap save skipped: {_e})")
    plt.show()
except Exception as e:
    log(f"(heatmap rendering skipped: {e})")

# --------------------------------------------------------------------------------
# 5) GATE G4 assert: effect must land in the EXPECTED region with the EXPECTED sign.
#    Expected region (Stolfo 2023 / Nikankin 2024): FINAL token position, middle-to-late
#    layers (back ~60% of the stack). Expected sign: POSITIVE recovery.
mid_late_start = int(np.floor(n_layers * 0.40))           # back ~60% of layers
final_col      = patch_recovery[:, final_pos]             # recovery vs layer at FINAL token
best_layer     = int(np.argmax(final_col))
best_recovery  = float(final_col[best_layer])
mid_late_peak  = float(np.max(final_col[mid_late_start:]))

# (a) peak recovery at the FINAL token must be strong and POSITIVE.
cond_strong   = best_recovery >= 0.5
# (b) peak must sit in the middle-to-late layer band (not only very early layers).
cond_band     = best_layer >= mid_late_start
# (c) FINAL token must dominate: its peak recovery beats best recovery at any NON-final pos.
nonfinal_peak = float(patch_recovery[:, :final_pos].max()) if final_pos > 0 else -np.inf
cond_lasttok  = best_recovery >= nonfinal_peak

g4_pass = bool(cond_strong and cond_band and cond_lasttok)
detail = (f"final-tok peak recovery={best_recovery:.2f} @ layer {best_layer} "
          f"(mid-late band starts @ {mid_late_start}); "
          f"mid-late final-tok peak={mid_late_peak:.2f}; "
          f"best non-final-pos recovery={nonfinal_peak:.2f}; "
          f"strong={cond_strong} in_band={cond_band} lasttok_dominates={cond_lasttok}")

print("---- G4 PIPELINE-VALIDATION CHECKS (target: Stolfo 2023 addition localization) ----")
print(f"  expected: LARGE +recovery at FINAL token (pos {final_pos}), middle-to-late layers")
print(f"  [a] strong final-token recovery (>=0.5):      {'PASS' if cond_strong else 'FAIL'} ({best_recovery:.2f})")
print(f"  [b] peak in middle-to-late layers (>= {mid_late_start}):  {'PASS' if cond_band else 'FAIL'} (layer {best_layer})")
print(f"  [c] final token dominates other positions:    {'PASS' if cond_lasttok else 'FAIL'} "
      f"({best_recovery:.2f} vs {nonfinal_peak:.2f})")
print(f"  ==> G4 {'PASS' if g4_pass else 'FAIL'}")

set_gate('G4', g4_pass, detail)

# Hard assert so a red G4 visibly blocks the cell (a green G4 is the license to trust
# later phases). Comment out the assert to render the sweep/heatmap for diagnosis only.
assert g4_pass, f"G4 FAILED — pipeline did not reproduce the known addition localization: {detail}"


---
# Plausibility verdict

The project is **PLAUSIBLE** when **G0–G4 are all green**. The two gates that decide whether the *science* is sound (not just the tooling) are **G2** (controlled stimulus — buildable with nothing but a tokenizer) and **G3** (the model genuinely computes the task). **G4** decides whether you can trust your own measurements.

Compute is never the plausibility constraint: the whole pre‑probe phase is a handful of cheap forward‑pass evaluations plus one known‑result reproduction. If every gate below is green, **Phase 6 (the depth‑1 padding‑invariance result) is the publishable spine** and the project is confirmed plausible.

The cell below reconstructs the consolidated gate status from disk (`gate_status.json`), so it reports correctly even after a GPU disconnect + restart.


In [ ]:
# ============================================================================
# Plausibility dashboard — consolidate G0..G4 from the on-disk gate ledger.
# Reconstructs from ART/gate_status.json, so it reports correctly even after a
# GPU disconnect + kernel restart (no in-memory state required).
# ============================================================================

def _read_gates():
    try:
        return get_gates()                       # checkpoint cell's ledger reader
    except Exception:
        for nm in ("gate_status", "gates"):
            try:
                if has_artifact(nm, "json"):
                    return load_json(nm)
            except Exception:
                pass
    return {}

_g = _read_gates()

def _ok(name):
    e = _g.get(name)
    if not e:
        return None
    return bool(e.get("passed", e.get("pass")))

def _detail(name):
    e = _g.get(name) or {}
    return e.get("detail", "")

_ROWS = [
    ("G_INFRA", "Infra  — checkpoint/resume + artifact round-trip", False),
    ("G0",      "Phase 0 — model loads + hooks (smoke test)",       True),
    ("G1",      "Phase 1 — novelty (MANUAL: see Phase 1 table)",    True),
    ("G2",      "Phase 2 — controlled stimulus (token-identical)",  True),
    ("G3",      "Phase 3 — model COMPUTES, engages operand, in-band",True),
    ("G4",      "Phase 5 — patching reproduces a KNOWN result",     True),
]

def _mark(v):
    return {True: "PASS", False: "FAIL", None: " -- "}[v]

print("=" * 74)
print("  OPERATOR-PRECEDENCE INTERPRETABILITY — PLAUSIBILITY DASHBOARD")
print("=" * 74)
for name, label, _core in _ROWS:
    v = _ok(name)
    d = _detail(name)
    d = (d[:60] + "...") if len(d) > 63 else d
    print(f"  [{_mark(v):>4}]  {label:<52}")
    if d:
        print(f"          └ {d}")
print("-" * 74)

# G1 is a manual/markdown gate (no set_gate call): treat 'not recorded' as a
# reminder to read the Phase 1 related-work table, not as a failure.
_core = ["G0", "G2", "G3", "G4"]
_core_vals = {g: _ok(g) for g in _core}
_core_pass = all(_core_vals[g] is True for g in _core)
_g1 = _ok("G1")
_g1_note = {True: "confirmed", False: "FAILED", None: "MANUAL — confirm via Phase 1 table"}[_g1]

# Surface the locked operand band + dataset sizes if those artifacts exist.
def _maybe(name, kind="json"):
    try:
        return load_json(name) if has_artifact(name, kind) else None
    except Exception:
        return None

_band = _maybe("locked_band_spec")
if _band:
    print(f"  Locked must-compute band : operands [{_band.get('operand_lo')}, "
          f"{_band.get('operand_hi')}]  (overall acc={_band.get('overall_accuracy')})")
_p2 = _maybe("phase2_stimuli")
if _p2 is not None:
    print(f"  Phase 2 experimental set : {len(_p2)} token-controlled records on disk")
print("-" * 74)

print(f"  Core gates (G0,G2,G3,G4) : {'ALL PASS' if _core_pass else 'NOT ALL PASS'}")
print(f"  Novelty gate (G1)        : {_g1_note}")
print()
if _core_pass and _g1 is not False:
    print("  VERDICT:  PLAUSIBLE  — G0/G2/G3/G4 green"
          + ("" if _g1 is True else " (pending manual G1 confirmation).") )
    print("            Phase 6 (depth-1 padding-invariance) is the publishable spine.")
else:
    missing = [g for g in _core if _core_vals[g] is not True]
    print(f"  VERDICT:  NOT YET PLAUSIBLE — unresolved core gate(s): {missing or ['G1']}")
    print("            Resolve the earliest failing/again-unrun gate before proceeding.")
    print("            (G2 & G3 failing = the SCIENCE is broken; G0/G4 = the TOOLING.)")
print("=" * 74)
